# 02 - Recording and Augmentation Pilot

A local, single-speaker experiment for **Hey GPT**. Run either the three code cells below or only the **Standalone control panel** cell at the end in a fresh project kernel. Install dependencies once with `uv sync`. **Run All only initializes controls; it does not record audio or create dataset folders.**

1. Click **Start positive**, say "Hey GPT" once after the microphone opens, and click **Stop positive** when finished. There is no countdown or preset recording duration.
2. The whole recording is selected initially. **No wake-word detection or automatic trimming is performed.** Use **Phrase (s)** only if you want to trim manually, and listen with **Preview selection**.
3. Choose **Window (s)** for the augmented output, then click **Generate 6 variants**. This duration does not control microphone capture. The selected phrase must fit the window; increase the window or trim manually if necessary. Audio is never silently truncated to fit.
4. Use **Start negative / Stop negative** for long background speech recordings, **Start hard negative / Stop hard negative** for confusable phrases or false triggers, or **Start noise / Stop noise** for mixing sources. All negative examples must exclude the complete wake phrase. Only one category records at a time.
5. **Augment all positives** uses saved manual selections or full positive recordings. For either negative category, Stop saves raw only; select a section in **Prepare negative samples**, click **Save samples**, then choose files under **Augment processed audio**. You can augment selected files or explicitly choose all processed files in Category. See the workflow below.

```text
data/
  positive/
    raw/          positive_0001.wav (native capture rate, float WAV)
    processed/    positive_0001.wav (full or manually selected phrase, mono 16 kHz PCM16)
    augmented/    positive_0001_run_0001/ (six fixed-window variants)
  negative/
    raw/          negative_0001.wav
    processed/    negative_0001_extract_0001/window_0001.wav
    augmented/    negative_0001_extract_0001_window_0001_run_0001/
  hard_negative/
    raw/          hard_negative_0001.wav
    processed/    hard_negative_0001_extract_0001/window_0001.wav
    augmented/    hard_negative_0001_extract_0001_window_0001_run_0001/
  noise/
    raw/          noise_0001.wav
```

Directories are created only when files are saved; filenames count upward across kernel restarts. All stages store **WAV**, with JSON metadata tracing derived files to their raw source. NumPy arrays are used in memory, not saved as `.npy`. Microphone recording streams mono RF64 float WAV to disk at the device's default sample rate and supports files larger than 4 GiB. Keep the kernel running until Stop finishes. Device or disk errors can also end capture; failed, active, and empty recordings are excluded from processing and mixing.

Existing positives can be reopened with the **Positive** dropdown. **Generate 6 variants** saves the current manual selection; each run creates a new folder. Batch augmentation uses the saved bounds, not an unsaved slider edit. Older intervals without a `selection_method: manual` marker may have come from the removed heuristic, so they default to the full recording when reopened. Original raw audio and existing augmented runs remain unchanged.

Noise is mixed directly from `noise/raw/`. Each noise variant randomly selects a completed file and an offset, reads the required segment, converts it to mono 16 kHz, and scales it to 10-25 dB SNR before mixing. Short tracks repeat from the beginning. Without eligible noise files, Gaussian noise is used for demonstration. Existing `noise/processed/` files are no longer used. Metadata records the chosen file, segment, and SNR. Noise playback after Stop previews the first five seconds. Negative and hard-negative playback uses the manually selected interval when Preview selection is clicked.

Preview logs, audio players, and waveforms update the Output widget directly rather than using notebook message capture. Re-running a panel closes its old child widgets. Playback preserves loudness unless the peak exceeds 1; only the playback copy is then attenuated. Audio stays local and is ignored by Git. No API is used.


In [ ]:
import asyncio
import json
import random
import shutil
from bisect import bisect_right
from datetime import datetime, timezone
from math import gcd
from pathlib import Path
from threading import Event, Thread
from uuid import uuid4

import ipywidgets as widgets
import numpy as np
import sounddevice as sd
import soundfile as sf
from audiomentations import AddBackgroundNoise, AddGaussianSNR, Gain, TimeStretch
from IPython.display import SVG, Audio, display
from scipy.signal import resample_poly

previous_recorder = globals().get("_recorder") or globals().get("_noise_recorder")
if previous_recorder is not None and previous_recorder.running:
    raise RuntimeError("Stop recording before rerunning setup or the standalone cell.")
previous_status_task = globals().get("_recording_status_task")
if previous_status_task is not None:
    previous_status_task.cancel()

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the GPTWake project.")

DATA_DIR = ROOT / "data"
SAMPLE_RATE = 16000
SPEAKER_ID = "self"
state = {"audio": None, "source": None, "updating_picker": False}
print("Dataset root:", DATA_DIR)
print(
    "Layout: positive, negative, and hard_negative have raw/processed/augmented; noise uses raw only."
)
print("No folders or recordings are created until you save data.")

In [ ]:
def log(*values):
    target = globals().get("output")
    if target is None:
        print(*values)
    else:
        target.append_stdout(" ".join(str(value) for value in values) + "\n")


def emit_preview(value):
    # Update widget state directly; do not also publish a captured IOPub display.
    target = globals().get("output")
    if target is None:
        display(value)
    else:
        target.append_display_data(value)


def close_panel(widget):
    for child in getattr(widget, "children", ()):
        close_panel(child)
    if hasattr(widget, "disabled"):
        widget.disabled = True
    widget.close()


def dataset_path(kind, stage):
    if kind not in {"positive", "negative", "hard_negative", "noise"} or stage not in {
        "raw",
        "processed",
        "augmented",
    }:
        raise ValueError("Unknown dataset category or stage.")
    return DATA_DIR / kind / stage


def next_path(directory, prefix, suffix=".wav"):
    # Inspect existing names on disk so restarting the kernel cannot overwrite a recording.
    number = 1
    for path in directory.glob(f"{prefix}_*"):
        tail = path.stem.rsplit("_", 1)[-1]
        if tail.isdigit():
            number = max(number, int(tail) + 1)
    return directory / f"{prefix}_{number:04d}{suffix}"


def write_metadata(path, metadata):
    path.write_text(
        json.dumps(
            metadata, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else str(x)
        ),
        encoding="utf-8",
    )


def write_wav(path, audio, sample_rate=SAMPLE_RATE, subtype="PCM_16", overwrite=True):
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim not in {1, 2} or not len(audio) or not np.isfinite(audio).all():
        raise ValueError("Expected nonempty, finite audio.")
    if sample_rate <= 0:
        raise ValueError("Sample rate must be positive.")
    peak = float(np.max(np.abs(audio)))
    if subtype != "FLOAT" and peak > 1:
        log(f"Reducing peak {peak:.3f} to avoid PCM clipping: {path.name}")
        audio = audio * (0.999 / peak)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb" if overwrite else "xb") as stream:
        sf.write(stream, audio, sample_rate, format="WAV", subtype=subtype)
    return audio


def normalize_source(path):
    audio, rate = sf.read(path, dtype="float32", always_2d=True)
    audio = audio.mean(axis=1)
    if rate != SAMPLE_RATE:
        divisor = gcd(rate, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // divisor, rate // divisor)
    return np.asarray(audio, dtype=np.float32)


def save_recording(kind, audio, sample_rate, metadata=None):
    path = next_path(dataset_path(kind, "raw"), kind)
    write_wav(path, audio, sample_rate, subtype="FLOAT", overwrite=False)
    write_metadata(
        path.with_suffix(".json"),
        {
            **(metadata or {}),
            "recording_id": path.stem,
            "category": kind,
            "label": {"positive": 1, "negative": 0, "hard_negative": 0, "noise": None}[kind],
            "speaker_id": SPEAKER_ID,
            "split": "unassigned",
            "raw_sample_rate": sample_rate,
            "created_at": datetime.now(timezone.utc).isoformat(),
        },
    )
    return path


def process_recording(kind, source, bounds=None, window_seconds=2.0):
    source = Path(source)
    if kind == "noise":
        return [source]
    sidecar = source.with_suffix(".json")
    source_metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
    if source_metadata.get("status", "complete") != "complete":
        raise ValueError("Only completed recordings can be processed.")
    if kind in {"negative", "hard_negative"}:
        raise ValueError(
            "Use the negative extraction panel to choose approved time intervals first."
        )
    audio = normalize_source(source)
    common = {
        "source": str(source.relative_to(ROOT)),
        "recording_id": source.stem,
        "category": kind,
        "label": {"positive": 1, "negative": 0, "hard_negative": 0, "noise": None}[kind],
        "sample_rate": SAMPLE_RATE,
        "speaker_id": source_metadata.get("speaker_id", SPEAKER_ID),
        "split": source_metadata.get("split", "unassigned"),
    }
    directory = dataset_path(kind, "processed")
    if kind == "positive":
        selection_method = "manual" if bounds is not None else "full_recording"
        if bounds is None:
            old_meta = directory / f"{source.stem}.json"
            saved = json.loads(old_meta.read_text()) if old_meta.exists() else {}
            # Legacy intervals may have been selected by the removed energy heuristic.
            if saved.get("selection_method") == "manual":
                bounds = saved["selection_seconds"]
                selection_method = "manual"
            else:
                bounds = (0.0, len(audio) / SAMPLE_RATE)
        common["selection_method"] = selection_method
        start, end = map(float, bounds)
        if not 0 <= start < end <= len(audio) / SAMPLE_RATE:
            raise ValueError("Phrase bounds must lie within the original recording.")
        segments = [
            (
                source.stem,
                audio[round(start * SAMPLE_RATE) : round(end * SAMPLE_RATE)],
                [start, end],
            )
        ]
    outputs = []
    for stem, segment, interval in segments:
        path = directory / f"{stem}.wav"
        write_wav(path, segment)
        write_metadata(path.with_suffix(".json"), {**common, "selection_seconds": interval})
        outputs.append(path)
    return outputs


def read_normalized_segment(source, start_frame, frame_count):
    with sf.SoundFile(source) as stream:
        rate = stream.samplerate
        stream.seek(start_frame)
        audio = stream.read(frame_count, dtype="float32", always_2d=True)
    if len(audio) != frame_count or not len(audio) or not np.isfinite(audio).all():
        raise ValueError("The selected segment is empty, incomplete, or contains invalid samples.")
    audio = audio.mean(axis=1)
    if rate != SAMPLE_RATE:
        divisor = gcd(rate, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // divisor, rate // divisor)
    expected = round(frame_count * SAMPLE_RATE / rate)
    return np.asarray(audio[:expected], dtype=np.float32)


def extract_negative_windows(
    kind,
    source,
    intervals,
    window_seconds=2.0,
    mode="random",
    count=20,
    stride_seconds=None,
    split=None,
    seed=None,
):
    if kind not in {"negative", "hard_negative"}:
        raise ValueError("Choose negative or hard_negative for window extraction.")
    source = Path(source)
    if source.resolve().parent != dataset_path(kind, "raw").resolve():
        raise ValueError("The selected source must belong to this category's raw directory.")
    sidecar = source.with_suffix(".json")
    metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
    if metadata.get("status", "complete") != "complete":
        raise ValueError("Only completed recordings can be processed.")
    info = sf.info(source)
    if not np.isfinite(window_seconds) or window_seconds <= 0:
        raise ValueError("Window duration must be positive and finite.")
    window_size = round(window_seconds * SAMPLE_RATE)
    native_size = round(window_size * info.samplerate / SAMPLE_RATE)
    if window_size < 1 or native_size < 1 or info.frames < 1:
        raise ValueError("Source and window must contain audio samples.")
    if mode not in {"random", "sliding", "clips", "placement"}:
        raise ValueError("Unknown window extraction mode.")
    if mode in {"clips", "placement"} and kind != "hard_negative":
        raise ValueError("Clip padding is intended for separately recorded hard negatives.")
    if not intervals:
        raise ValueError("Enter at least one approved start,end interval in seconds.")
    ranges = []
    for interval in intervals:
        if len(interval) != 2:
            raise ValueError("Each interval needs a start and end in seconds.")
        start, end = map(float, interval)
        if not (np.isfinite(start) and np.isfinite(end) and 0 <= start < end <= info.duration):
            raise ValueError("Each interval must lie within the source duration.")
        # Quantize inward: a window must never read outside an approved interval.
        first = int(np.ceil(start * info.samplerate))
        last = info.frames if end == info.duration else int(np.floor(end * info.samplerate))
        if last <= first or round((last - first) * SAMPLE_RATE / info.samplerate) < 1:
            raise ValueError(
                "An interval is shorter than one audio sample at the source or target rate."
            )
        if mode in {"clips", "placement"} and last - first > native_size:
            raise ValueError(
                "Hard-negative clip exceeds the window; trim manually or enlarge the window."
            )
        if mode not in {"clips", "placement"} and last - first < native_size:
            raise ValueError("Every interval must contain at least one complete window.")
        ranges.append((first, last))
    ranges.sort()
    if any(current[0] < previous[1] for previous, current in zip(ranges, ranges[1:])):
        raise ValueError("Approved intervals must not overlap.")
    split = metadata.get("split", "unassigned") if split is None else split
    if split not in {"unassigned", "train", "validation", "test"}:
        raise ValueError("Unknown dataset split.")
    if "window_split" in metadata and metadata["window_split"] != split:
        raise ValueError(
            f"This source is locked to split {metadata['window_split']}; keep the same split."
        )
    seed = uuid4().int % (2**32) if seed is None else int(seed)
    if mode in {"random", "placement"}:
        if not isinstance(count, (int, np.integer)) or isinstance(count, bool) or count < 1:
            raise ValueError("Sample count must be a positive integer.")
    if mode == "random":
        lengths = [last - first - native_size + 1 for first, last in ranges]
        cumulative = np.cumsum(lengths).tolist()
        if count > cumulative[-1]:
            raise ValueError("Requested count exceeds the number of unique window positions.")
        # Sampling an integer range avoids allocating an array for an hours-long track.
        positions = random.Random(seed).sample(range(cumulative[-1]), count)
        selected = []
        for position in positions:
            index = bisect_right(cumulative, position)
            offset = position - (cumulative[index - 1] if index else 0)
            start = ranges[index][0] + offset
            selected.append((start, start + native_size))
    elif mode == "sliding":
        stride_seconds = window_seconds if stride_seconds is None else stride_seconds
        if not np.isfinite(stride_seconds) or stride_seconds <= 0:
            raise ValueError("Sliding stride must be positive and finite.")
        stride = round(stride_seconds * info.samplerate)
        if stride < 1:
            raise ValueError("Sliding stride must be at least one source sample.")
        selected = (
            (start, start + native_size)
            for first, last in ranges
            for start in range(first, last - native_size + 1, stride)
        )
    else:
        selected = iter(ranges)
    relative = str(source.relative_to(ROOT))
    folder = next_path(dataset_path(kind, "processed"), source.stem + "_extract", suffix="")
    source_group = metadata.get("source_group", relative)
    metadata.update(split=split, window_split=split, source_group=source_group)
    write_metadata(sidecar, metadata)
    outputs = []
    placement_rng = np.random.default_rng(seed) if mode == "placement" else None
    for start, end in selected:
        audio = read_normalized_segment(source, start, end - start)
        if not len(audio):
            raise ValueError("Selected clip is shorter than one target sample.")
        if mode in {"clips", "placement"} and len(audio) > window_size:
            raise ValueError(
                "The normalized clip exceeds the output window; no audio was truncated."
            )
        # Complete-window crops may differ by one sample due to resampling rounding.
        audio = audio[:window_size]
        for copy_index in range(count if mode == "placement" else 1):
            slack = window_size - len(audio)
            if mode == "placement":
                left = int(placement_rng.integers(0, slack + 1))
            else:
                left = slack // 2 if mode == "clips" else 0
            right = slack - left
            placed = np.pad(audio, (left, right))
            path = folder / f"window_{len(outputs) + 1:04d}.wav"
            write_wav(path, placed, overwrite=False)
            write_metadata(
                path.with_suffix(".json"),
                {
                    "source": relative,
                    "recording_id": source.stem,
                    "category": kind,
                    "label": 0,
                    "sample_rate": SAMPLE_RATE,
                    "speaker_id": metadata.get("speaker_id", SPEAKER_ID),
                    "split": split,
                    "source_group": source_group,
                    "selection_seconds": [start / info.samplerate, end / info.samplerate],
                    "source_start_frame": start,
                    "source_end_frame": end,
                    "source_sample_rate": info.samplerate,
                    "window_seconds": window_size / SAMPLE_RATE,
                    "windowing_mode": mode,
                    "sampling_seed": seed if mode in {"random", "placement"} else None,
                    "placement_offset_samples": left if mode == "placement" else None,
                    "placement_offset_seconds": left / SAMPLE_RATE if mode == "placement" else None,
                    "padding_left_samples": left,
                    "padding_right_samples": right,
                },
            )
            outputs.append(path)
    write_metadata(
        folder / "extraction.json",
        {
            "source": relative,
            "source_group": source_group,
            "category": kind,
            "label": 0,
            "split": split,
            "mode": mode,
            "window_seconds": window_size / SAMPLE_RATE,
            "approved_intervals_seconds": [
                [a / info.samplerate, b / info.samplerate] for a, b in ranges
            ],
            "seed": seed if mode in {"random", "placement"} else None,
            "copies_per_clip": count if mode == "placement" else None,
            "stride_seconds": stride / info.samplerate if mode == "sliding" else None,
            "window_count": len(outputs),
        },
    )
    log("Saved", len(outputs), kind, "windows:", folder.relative_to(ROOT))
    return outputs


def processed_negative_windows(kind):
    if kind not in {"negative", "hard_negative"}:
        raise ValueError("Choose a negative category.")
    outputs = []
    for path in sorted(dataset_path(kind, "processed").rglob("*.wav")):
        sidecar = path.with_suffix(".json")
        if not sidecar.exists():
            continue
        metadata = json.loads(sidecar.read_text())
        if metadata.get("category") == kind and metadata.get("label") == 0:
            outputs.append(path)
    return outputs


def prepare_noise_library():
    # Load only the segment needed for mixing; long tracks are never normalized in full.
    return dataset_path("noise", "raw")


def available_audio_files(directory):
    candidates = []
    for path in sorted(directory.glob("*.wav")):
        sidecar = path.with_suffix(".json")
        try:
            metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
            if metadata.get("status", "complete") != "complete":
                continue
            if sf.info(path).frames > 0:
                candidates.append(path)
        except (OSError, ValueError, RuntimeError):
            continue
    return candidates


def make_noise_transform(noise_dir):
    candidates = available_audio_files(noise_dir)
    if candidates:
        return AddBackgroundNoise(sounds_path=candidates, min_snr_db=10, max_snr_db=25, p=1)
    return AddGaussianSNR(min_snr_db=10, max_snr_db=25, p=1)


def augment_recording(kind, processed_source, window_seconds=2.0, show_players=False):
    if kind not in {"positive", "negative", "hard_negative"}:
        raise ValueError("Only positive, negative, and hard_negative examples are augmented here.")
    processed_source = Path(processed_source)
    audio, rate = sf.read(processed_source, dtype="float32")
    if rate != SAMPLE_RATE or audio.ndim != 1:
        raise ValueError("Process the source into mono 16 kHz audio first.")
    metadata = json.loads(processed_source.with_suffix(".json").read_text())
    seed = uuid4().int % (2**32)
    random.seed(seed)
    noise_dir = prepare_noise_library()
    if kind == "positive":
        variants = make_variants(audio, window_seconds, noise_dir, seed)
    else:
        # Negative windows already have fixed length; avoid three identical placement copies.
        quiet = Gain(min_gain_db=-9, max_gain_db=-3, p=1)
        stretch = TimeStretch(min_rate=1.03, max_rate=1.10, leave_length_unchanged=True, p=1)
        noise = make_noise_transform(noise_dir)
        variants = []
        for name, transform in [("quieter", quiet), ("noise", noise), ("faster", stretch)]:
            result = transform(samples=audio.copy(), sample_rate=SAMPLE_RATE)
            variants.append((name, result, dict(transform.parameters)))
    prefix = processed_source.stem
    if kind in {"negative", "hard_negative"} and processed_source.parent != dataset_path(
        kind, "processed"
    ):
        prefix = processed_source.parent.name + "_" + prefix
    folder = next_path(dataset_path(kind, "augmented"), prefix + "_run", suffix="")
    entries = []
    for name, result, parameters in variants:
        result = write_wav(folder / f"{name}.wav", result, overwrite=False)
        parameters = dict(parameters)
        if "noise_file_path" in parameters:
            parameters["noise_file_path"] = str(
                Path(parameters["noise_file_path"]).relative_to(ROOT)
            )
        entries.append({"file": f"{name}.wav", "parameters": parameters})
        if show_players:
            show_audio(result, name)
    write_metadata(
        folder / "metadata.json",
        {
            **metadata,
            "processed_source": str(processed_source.relative_to(ROOT)),
            "window_seconds": len(variants[0][1]) / SAMPLE_RATE,
            "augmentation_seed": seed,
            "variants": entries,
        },
    )
    log("Saved", len(variants), "variants:", folder.relative_to(ROOT))
    return folder


def import_legacy_recordings():
    # Copy old sources without deleting or re-encoding them. Repeated imports are safe.
    imported = []
    patterns = {
        "positive": "raw/positive_pilot/**/*.wav",
        "noise": "negatives/room_noise_pilot/**/*.wav",
    }
    for kind, pattern in patterns.items():
        known = {
            json.loads(path.read_text()).get("legacy_source")
            for path in dataset_path(kind, "raw").glob("*.json")
        }
        for source in sorted(DATA_DIR.glob(pattern)):
            relative = str(source.relative_to(ROOT))
            if relative in known:
                continue
            destination = next_path(dataset_path(kind, "raw"), kind)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with source.open("rb") as src, destination.open("xb") as dst:
                shutil.copyfileobj(src, dst)
            old_sidecar = source.with_suffix(".json")
            write_metadata(
                destination.with_suffix(".json"),
                {
                    "recording_id": destination.stem,
                    "category": kind,
                    "speaker_id": SPEAKER_ID,
                    "split": "unassigned",
                    "legacy_source": relative,
                    "raw_sample_rate": sf.info(destination).samplerate,
                    "original_metadata": json.loads(old_sidecar.read_text())
                    if old_sidecar.exists()
                    else {},
                },
            )
            process_recording(kind, destination)
            known.add(relative)
            imported.append(destination)
    return imported


def show_audio(audio, title):
    peak = float(np.max(np.abs(audio)))
    log(f"{title}: {len(audio) / SAMPLE_RATE:.2f}s, peak={peak:.3f}")
    playback = audio
    if peak > 1:
        # Float capture and resampling can exceed the player's [-1, 1] range.
        playback = audio * (0.999 / peak)
        log("Preview volume reduced to fit playback range; source audio is unchanged.")
    emit_preview(Audio(playback, rate=SAMPLE_RATE, normalize=False))
    # Display every bin's peak so short transients remain visible.
    bins = np.array_split(np.abs(audio), min(500, len(audio)))
    peaks = [float(b.max()) for b in bins]
    lines = "".join(
        f'<line x1="{i * 1.2:.1f}" x2="{i * 1.2:.1f}" '
        f'y1="{45 - v * 42:.1f}" y2="{45 + v * 42:.1f}" '
        'stroke="#2475c0"/>'
        for i, v in enumerate(peaks)
    )
    emit_preview(
        SVG(
            f'<svg xmlns="http://www.w3.org/2000/svg" width="600" height="90" '
            f'viewBox="0 0 600 90"><rect width="600" height="90" fill="#f3f6fa"/>{lines}</svg>'
        )
    )


def random_place(phrase, window_seconds, rng):
    total = round(window_seconds * SAMPLE_RATE)
    if not 0 < len(phrase) <= total:
        raise ValueError(
            "The selected phrase does not fit. Adjust its bounds or increase the window."
        )
    offset = int(rng.integers(0, total - len(phrase) + 1))
    result = np.zeros(total, dtype=np.float32)
    result[offset : offset + len(phrase)] = phrase
    return result, offset / SAMPLE_RATE


def make_variants(phrase, window_seconds, noise_dir, seed):
    rng = np.random.default_rng(seed)
    if len(phrase) > round(window_seconds * SAMPLE_RATE):
        raise ValueError(
            "Phrase exceeds the window; no positive was truncated. Adjust selection or window."
        )
    variants = []
    for i in range(3):
        audio, offset = random_place(phrase, window_seconds, rng)
        variants.append((f"placement_{i + 1}", audio, {"offset_seconds": offset}))
    quiet = Gain(min_gain_db=-9, max_gain_db=-3, p=1)
    quieter = quiet(samples=phrase.copy(), sample_rate=SAMPLE_RATE)
    audio, offset = random_place(quieter, window_seconds, rng)
    variants.append(("quieter", audio, {"offset_seconds": offset, **quiet.parameters}))
    audio, offset = random_place(phrase, window_seconds, rng)
    noise = make_noise_transform(noise_dir)
    label = "room_noise" if isinstance(noise, AddBackgroundNoise) else "gaussian_noise_demo"
    noisy = noise(samples=audio.copy(), sample_rate=SAMPLE_RATE)
    variants.append((label, noisy, {"offset_seconds": offset, **noise.parameters}))
    # Faster speech fits whenever the original phrase fits. Preserve pitch and do not pad/crop it here.
    stretch = TimeStretch(min_rate=1.03, max_rate=1.10, leave_length_unchanged=False, p=1)
    faster = stretch(samples=phrase.copy(), sample_rate=SAMPLE_RATE)
    audio, offset = random_place(faster, window_seconds, rng)
    variants.append(("faster", audio, {"offset_seconds": offset, **stretch.parameters}))
    return variants


class AudioRecorder:
    """Capture native-rate blocks to disk while the notebook remains interactive."""

    def __init__(self):
        self.path = None
        self.kind = None
        self.frames = 0
        self.rate = 0
        self.error = None
        self.overflows = 0
        self.ready = Event()
        self._stop = Event()
        self._thread = None
        self._stream = None
        self._metadata = {}

    @property
    def running(self):
        return self._thread is not None and self._thread.is_alive()

    def start(self, device, kind="noise"):
        if self.running:
            raise RuntimeError("A recording is already running.")
        if kind not in {"positive", "negative", "hard_negative", "noise"}:
            raise ValueError("Unknown recording category.")
        info = sd.query_devices(device, "input")
        rate = int(info["default_samplerate"])
        sd.check_input_settings(device=device, channels=1, samplerate=rate, dtype="float32")
        if not sf.check_format("RF64", "FLOAT"):
            raise RuntimeError("This libsndfile installation does not support RF64 float WAV.")
        path = next_path(dataset_path(kind, "raw"), kind)
        path.parent.mkdir(parents=True, exist_ok=True)
        handle = path.open("xb")
        self.path, self.rate, self.kind = path, rate, kind
        self.frames, self.overflows, self.error = 0, 0, None
        self.ready.clear()
        self._stop.clear()
        self._metadata = {
            "recording_id": path.stem,
            "category": kind,
            "label": {"positive": 1, "negative": 0, "hard_negative": 0, "noise": None}[kind],
            "speaker_id": SPEAKER_ID,
            "split": "unassigned",
            "device": str(info["name"]),
            "raw_sample_rate": rate,
            "capture_rate": rate,
            "format": "RF64",
            "subtype": "FLOAT",
            "created_at": datetime.now(timezone.utc).isoformat(),
            "status": "recording",
        }
        try:
            write_metadata(path.with_suffix(".json"), self._metadata)
            self._thread = Thread(target=self._record, args=(handle, device), daemon=True)
            self._thread.start()
        except Exception:
            handle.close()
            raise
        return path

    def _record(self, handle, device):
        try:
            # RF64 keeps the .wav extension and can exceed the ordinary WAV 4 GiB limit.
            with handle:
                with sf.SoundFile(
                    handle,
                    mode="w",
                    samplerate=self.rate,
                    channels=1,
                    format="RF64",
                    subtype="FLOAT",
                ) as sink:
                    with sd.InputStream(
                        device=device,
                        samplerate=self.rate,
                        channels=1,
                        dtype="float32",
                        blocksize=4096,
                    ) as stream:
                        self._stream = stream
                        self.ready.set()
                        while not self._stop.is_set():
                            block, overflowed = stream.read(4096)
                            if len(block):
                                sink.write(block)
                                sink.flush()
                                self.frames += len(block)
                            self.overflows += int(overflowed)
        except Exception as error:
            self.error = error
        finally:
            self._stream = None
            self._metadata.update(
                status="failed" if self.error else ("complete" if self.frames else "empty"),
                frames=self.frames,
                duration_seconds=self.frames / self.rate,
                overflow_count=self.overflows,
                stopped_at=datetime.now(timezone.utc).isoformat(),
                error=str(self.error) if self.error else None,
            )
            try:
                write_metadata(self.path.with_suffix(".json"), self._metadata)
            except Exception as error:
                self.error = error
            self.ready.set()

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2)
            if self.running and self._stream is not None:
                self._stream.abort()
                self._thread.join(timeout=2)
            if self.running:
                raise RuntimeError(
                    "Microphone has not stopped yet. Keep the kernel open and retry Stop."
                )
        return self.path


def preview_recording(path, seconds=5):
    # Preview a bounded prefix rather than reading an hours-long recording into memory.
    info = sf.info(path)
    audio, rate = sf.read(
        path,
        frames=min(info.frames, round(seconds * info.samplerate)),
        dtype="float32",
        always_2d=True,
    )
    if not len(audio):
        return
    audio = audio.mean(axis=1)
    if rate != SAMPLE_RATE:
        factor = gcd(rate, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // factor, rate // factor)
    show_audio(
        audio.astype(np.float32), f"Recording preview: first {min(seconds, info.duration):.1f}s"
    )

In [ ]:
previous_recorder = globals().get("_recorder") or globals().get("_noise_recorder")
if previous_recorder is not None and previous_recorder.running:
    raise RuntimeError("Stop recording before rerunning the control panel.")
previous_status_task = globals().get("_recording_status_task")
if previous_status_task is not None:
    previous_status_task.cancel()
previous_panel = globals().get("_pilot_panel")
if previous_panel is not None:
    close_panel(previous_panel)
_recorder = AudioRecorder()
_recording_status_task = None

input_options = [("System default microphone", None)] + [
    (f"{i}: {d['name']}", i)
    for i, d in enumerate(sd.query_devices())
    if d["max_input_channels"] > 0
]
device = widgets.Dropdown(
    options=input_options, description="Microphone:", layout={"width": "650px"}
)
window = widgets.Dropdown(options=[1.5, 2.0, 2.5, 3.0], value=2.0, description="Window (s):")
source_picker = widgets.Dropdown(description="Positive:", layout={"width": "650px"})
selection = widgets.FloatRangeSlider(
    value=(0, 1.0),
    min=0,
    max=1.0,
    step=0.01,
    description="Phrase (s):",
    readout_format=".2f",
    continuous_update=False,
    layout={"width": "650px"},
)
record_button = widgets.Button(description="Start positive", button_style="success")
negative_button = widgets.Button(description="Start negative")
hard_negative_button = widgets.Button(description="Start hard negative", layout={"width": "180px"})
hard_negative_stop_button = widgets.Button(
    description="Stop hard negative",
    button_style="danger",
    disabled=True,
    layout={"width": "180px"},
)
record_stop_button = widgets.Button(
    description="Stop positive", button_style="danger", disabled=True
)
negative_stop_button = widgets.Button(
    description="Stop negative", button_style="danger", disabled=True
)
noise_button = widgets.Button(description="Start noise", button_style="success")
noise_stop_button = widgets.Button(description="Stop noise", button_style="danger", disabled=True)
recording_status = widgets.HTML(value="Recorder idle. Click Start, then Stop when finished.")
preview_button = widgets.Button(description="Preview selection")
augment_button = widgets.Button(description="Generate 6 variants", button_style="info")
batch_button = widgets.Button(description="Augment all positives")
negative_aug_button = widgets.Button(description="Augment selected", layout={"width": "220px"})
negative_aug_scope = widgets.Dropdown(
    options=[("Selected processed files", "selected"), ("All processed files in Category", "all")],
    value="selected",
    description="Augment:",
    layout={"width": "550px"},
)
negative_processed = widgets.SelectMultiple(
    options=[],
    value=(),
    description="Files:",
    rows=7,
    layout={"width": "750px"},
)
negative_processed_refresh = widgets.Button(description="Refresh file list")
negative_aug_summary = widgets.HTML()
negative_window = widgets.Dropdown(
    options=window.options, value=window.value, description="Output (s):"
)
negative_summary = widgets.HTML()
negative_multiple = widgets.Checkbox(value=False, description="Use several time intervals")
negative_split_info = widgets.HTML(value="Split is locked for this source after the first save.")
negative_kind = widgets.Dropdown(
    options=[("Negative", "negative"), ("Hard negative", "hard_negative")], description="Category:"
)
negative_source = widgets.Dropdown(description="Source:", layout={"width": "650px"})
negative_info = widgets.HTML(value="Choose a completed negative recording.")
negative_ranges = widgets.Textarea(
    description="Ranges (s):",
    placeholder="One start,end pair per line, e.g. 10,30",
    layout={"width": "650px", "height": "90px"},
)
negative_mode = widgets.Dropdown(
    options=[("Random crops from recording", "random"), ("Slide through selection", "sliding")],
    description="Method:",
    layout={"width": "450px"},
)
negative_count = widgets.BoundedIntText(value=20, min=1, max=10000, description="Samples:")
negative_stride = widgets.FloatText(value=2.0, description="Stride (s):")
negative_split = widgets.Dropdown(
    options=["unassigned", "train", "validation", "test"], description="Split:"
)
negative_selection = widgets.FloatRangeSlider(
    value=(0, 0),
    min=0,
    max=1,
    step=0.01,
    description="Selection (s):",
    readout_format=".2f",
    continuous_update=False,
    disabled=True,
    layout={"width": "650px"},
    style={"description_width": "100px"},
)
negative_preview_button = widgets.Button(description="Preview selection", layout={"width": "180px"})
negative_use_range_button = widgets.Button(
    description="Replace interval list", layout={"width": "180px"}
)
negative_add_range_button = widgets.Button(
    description="Add selection to list", layout={"width": "180px"}
)
extract_button = widgets.Button(
    description="Save samples", button_style="info", layout={"width": "200px"}
)
legacy_button = widgets.Button(description="Import old recordings")
output = widgets.Output()
buttons = [
    record_button,
    record_stop_button,
    negative_button,
    negative_stop_button,
    hard_negative_button,
    hard_negative_stop_button,
    noise_button,
    noise_stop_button,
    preview_button,
    augment_button,
    batch_button,
    negative_aug_button,
    negative_processed_refresh,
    negative_preview_button,
    negative_use_range_button,
    negative_add_range_button,
    extract_button,
    legacy_button,
]


all_controls = [
    *buttons,
    device,
    window,
    source_picker,
    selection,
    negative_kind,
    negative_source,
    negative_ranges,
    negative_mode,
    negative_count,
    negative_stride,
    negative_split,
    negative_selection,
    negative_window,
    negative_multiple,
    negative_aug_scope,
    negative_processed,
]
negative_picker_state = {"updating": False}
negative_ui_state = {"updating": False}


stop_buttons = {
    "positive": record_stop_button,
    "negative": negative_stop_button,
    "hard_negative": hard_negative_stop_button,
    "noise": noise_stop_button,
}


def update_controls():
    recording = _recorder.running
    for control in all_controls:
        control.disabled = recording
    for control in [
        negative_selection,
        negative_preview_button,
        negative_use_range_button,
        negative_add_range_button,
    ]:
        control.disabled = recording or negative_source.value is None
    update_negative_preparation()
    update_processed_augmentation()
    for kind, button in stop_buttons.items():
        button.disabled = not (recording and _recorder.kind == kind)


def run_action(action):
    for control in all_controls:
        control.disabled = True
    output.outputs = ()
    try:
        action()
    except Exception as error:
        log(f"{type(error).__name__}: {error}")
    finally:
        update_controls()


async def monitor_recording():
    while _recorder.running:
        if _recorder.ready.is_set():
            elapsed = _recorder.frames / _recorder.rate
            recording_status.value = (
                f"Recording {_recorder.kind}: {elapsed:.1f} seconds. "
                f"Click Stop {_recorder.kind} to finish."
            )
        await asyncio.sleep(0.25)
    update_controls()
    if _recorder.error:
        recording_status.value = "Recording failed; the partial file is excluded from processing."
        log("Recording error:", _recorder.error)


def start_recording(kind):
    global _recording_status_task
    if _recording_status_task is not None:
        _recording_status_task.cancel()
    path = _recorder.start(device.value, kind)
    recording_status.value = (
        f"Opening microphone for {kind}; no countdown. Click Stop {kind} to finish."
    )
    log("Recording to:", path.relative_to(ROOT))
    log("Click Stop", kind, "when finished. There is no preset duration.")
    if kind == "positive":
        log("Say the complete Hey GPT phrase once. No automatic trimming is applied.")
    elif kind == "hard_negative":
        log(
            "Record a similar-sounding phrase or known false trigger, without the complete wake phrase."
        )
    else:
        log("Avoid saying the complete wake phrase in negative or noise recordings.")
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        pass
    else:
        _recording_status_task = loop.create_task(monitor_recording())


def stop_recording():
    global _recording_status_task
    if _recording_status_task is not None:
        _recording_status_task.cancel()
    path = _recorder.stop()
    duration = _recorder.frames / _recorder.rate
    kind = _recorder.kind
    recording_status.value = f"{kind.title()} saved: {duration:.2f} seconds."
    log("Saved original:", path.relative_to(ROOT))
    if _recorder.error:
        recording_status.value = "Recording failed; the partial file is excluded from processing."
        log("Recording error:", _recorder.error)
        return
    if _recorder.frames == 0:
        log("No audio was captured; this empty recording is excluded from processing.")
        return
    if _recorder.overflows:
        log(
            "Warning: input overflow occurred",
            _recorder.overflows,
            "time(s); review the recording.",
        )
    if kind == "positive":
        refresh_sources(path)
        load_positive(path)
    elif kind in {"negative", "hard_negative"}:
        negative_kind.value = kind
        refresh_negative_sources(path)
        log("Raw saved. Drag Selection, preview it, then click Save samples.")
        log("Drag Selection and click Preview selection to listen to the chosen interval.")
    else:
        preview_recording(path)


def refresh_sources(selected=None):
    state["updating_picker"] = True
    try:
        sources = available_audio_files(dataset_path("positive", "raw"))
        source_picker.options = [("Select a recording", None)] + [(p.name, str(p)) for p in sources]
        if selected is not None:
            source_picker.value = str(selected)
    finally:
        state["updating_picker"] = False


def load_positive(source):
    source = Path(source)
    audio = normalize_source(source)
    processed = process_recording("positive", source)[0]
    metadata = json.loads(processed.with_suffix(".json").read_text())
    state.update(audio=audio, source=source)
    selection.value = (0, 0)
    selection.max = len(audio) / SAMPLE_RATE
    selection.value = metadata["selection_seconds"]
    log("Raw source:", source.relative_to(ROOT))
    log("Processed phrase:", processed.relative_to(ROOT))
    show_audio(audio, "Original timing, normalized for preview")
    log(
        "The full recording is selected unless you saved a manual selection. Adjust Phrase only if needed."
    )


def selected_phrase():
    if state["audio"] is None:
        raise ValueError("Record or select a positive sample first.")
    start, end = selection.value
    phrase = state["audio"][round(start * SAMPLE_RATE) : round(end * SAMPLE_RATE)]
    if not len(phrase):
        raise ValueError("Select a nonempty phrase interval.")
    return phrase


def generate_variants():
    phrase = selected_phrase()
    if len(phrase) > round(window.value * SAMPLE_RATE):
        raise ValueError(
            "Phrase exceeds the window. Adjust the selection or increase the window; no audio was truncated."
        )
    processed = process_recording("positive", state["source"], bounds=selection.value)[0]
    show_audio(phrase, "Selected phrase")
    augment_recording("positive", processed, window_seconds=window.value, show_players=True)


def augment_all(kind):
    if kind in {"negative", "hard_negative"}:
        outputs = processed_negative_windows(kind)
        if not outputs:
            raise ValueError(
                "No saved samples found. Select a source and click Save samples first."
            )
        for processed in outputs:
            augment_recording(kind, processed)
        log(
            "Augmented",
            len(outputs),
            "existing",
            kind,
            "windows; no raw recording was reprocessed.",
        )
        return
    sources = available_audio_files(dataset_path(kind, "raw"))
    if not sources:
        raise ValueError(f"No {kind} recordings found.")
    count = 0
    for source in sources:
        outputs = process_recording(kind, source, window_seconds=window.value)
        for processed in outputs:
            if sf.info(processed).frames > round(window.value * SAMPLE_RATE):
                log(
                    "Skipped oversized phrase:",
                    source.name,
                    "- review its bounds or increase the window.",
                )
                continue
            augment_recording(kind, processed, window_seconds=window.value)
            count += 1
    log("Augmented", count, "processed positive file(s). Saved selections were reused.")


def refresh_processed_samples(selected=None):
    paths = processed_negative_windows(negative_kind.value)
    previous = negative_processed.value if selected is None else tuple(str(p) for p in selected)
    base = dataset_path(negative_kind.value, "processed")
    negative_processed.options = [(str(p.relative_to(base)), str(p)) for p in paths]
    allowed = {str(p) for p in paths}
    negative_processed.value = tuple(value for value in previous if value in allowed)
    if selected is not None:
        negative_aug_scope.value = "selected"
    update_processed_augmentation()


def update_processed_augmentation(change=None):
    recording = _recorder.running
    available = len(negative_processed.options)
    selected = len(negative_processed.value)
    all_files = negative_aug_scope.value == "all"
    count = available if all_files else selected
    negative_processed.disabled = recording or all_files
    negative_processed_refresh.disabled = recording
    negative_aug_button.disabled = recording or count == 0
    negative_aug_button.description = (
        f"Augment all {count} files" if all_files else f"Augment {count} selected"
    )
    if all_files:
        negative_aug_summary.value = f"All {available} processed files in {negative_kind.value}, across every raw source. Each click creates new augmentation runs."
    else:
        negative_aug_summary.value = f"{selected} of {available} processed files selected. Only highlighted files will be augmented. Save samples selects the newly saved files automatically."


def augment_chosen_negatives():
    kind = negative_kind.value
    available = {str(p): p for p in processed_negative_windows(kind)}
    if negative_aug_scope.value == "all":
        chosen = list(available.values())
    else:
        selected = negative_processed.value
        if any(value not in available for value in selected):
            raise ValueError(
                "A selected file is no longer available. Refresh the file list and select again."
            )
        chosen = [available[value] for value in selected]
    if not chosen:
        raise ValueError(
            "Select at least one processed file, or choose All processed files in Category."
        )
    for path in chosen:
        augment_recording(kind, path)
    log("Augmented", len(chosen), kind, "processed file(s).")


def refresh_negative_sources(selected=None):
    negative_picker_state["updating"] = True
    try:
        sources = available_audio_files(dataset_path(negative_kind.value, "raw"))
        negative_source.options = [("Select a recording", None)] + [
            (p.name, str(p)) for p in sources
        ]
        if selected is not None:
            negative_source.value = str(selected)
    finally:
        negative_picker_state["updating"] = False
    load_negative_source()


def load_negative_source():
    negative_ui_state["updating"] = True
    negative_ranges.value = ""
    negative_multiple.value = False
    negative_selection.value = (0, 0)
    negative_selection.max = 1
    if negative_source.value is None:
        negative_info.value = "Choose a completed negative recording."
        negative_split.value = "unassigned"
        negative_ui_state["updating"] = False
        update_controls()
        return
    source = Path(negative_source.value)
    info = sf.info(source)
    sidecar = source.with_suffix(".json")
    metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
    negative_split.value = metadata.get("split", "unassigned")
    negative_info.value = (
        f"Recording: {info.duration:.2f} seconds. Drag the handles to choose a section."
    )
    negative_selection.max = info.duration
    negative_selection.value = (0, info.duration)
    negative_mode.value = "placement" if negative_kind.value == "hard_negative" else "random"
    negative_count.max = 10000
    negative_count.value = 1 if negative_kind.value == "hard_negative" else 20
    negative_ui_state["updating"] = False
    log("Selected", source.name, f"({info.duration:.3f}s).")
    update_controls()


def change_negative_category(change):
    negative_mode.options = (
        [
            ("Random position in output window", "placement"),
            ("Centered in output window", "clips"),
        ]
        if change["new"] == "hard_negative"
        else [("Random crops from recording", "random"), ("Slide through selection", "sliding")]
    )
    negative_mode.value = "placement" if change["new"] == "hard_negative" else "random"
    refresh_negative_sources()
    refresh_processed_samples(selected=[])


def parse_negative_ranges():
    intervals = []
    for line in negative_ranges.value.splitlines():
        if not line.strip():
            continue
        try:
            start, end = map(float, line.split(","))
        except ValueError as error:
            raise ValueError(
                "Use one start,end pair in seconds per line, for example 10,30."
            ) from error
        intervals.append((start, end))
    return intervals


def negative_extraction_intervals():
    if negative_multiple.value:
        return parse_negative_ranges()
    path, info, start, end, first, last = selected_negative_range()
    return [(start, end)]


def extract_selected_negatives():
    if negative_source.value is None:
        raise ValueError("Select a negative source first.")
    saved = extract_negative_windows(
        negative_kind.value,
        negative_source.value,
        negative_extraction_intervals(),
        window_seconds=window.value,
        mode=negative_mode.value,
        count=negative_count.value,
        stride_seconds=negative_stride.value,
        split=negative_split.value,
    )

    refresh_processed_samples(selected=saved)


def update_negative_preparation(change=None):
    if negative_ui_state["updating"]:
        return
    negative_ui_state["updating"] = True
    try:
        recording = _recorder.running
        method = negative_mode.value
        negative_count.layout.display = "" if method in {"random", "placement"} else "none"
        negative_count.description = "Copies / clip:" if method == "placement" else "Samples:"
        negative_count.style.description_width = "100px" if method == "placement" else "initial"
        if method == "placement":
            negative_count.max = 10000
        negative_stride.layout.display = "" if method == "sliding" else "none"
        negative_ranges_box.layout.display = "" if negative_multiple.value else "none"
        for control in [negative_ranges, negative_use_range_button, negative_add_range_button]:
            control.disabled = (
                recording or negative_source.value is None or not negative_multiple.value
            )
        try:
            if negative_source.value is None:
                raise ValueError("Choose a recording above to prepare samples.")
            info = sf.info(negative_source.value)
            sidecar = Path(negative_source.value).with_suffix(".json")
            source_metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
            locked_split = source_metadata.get("window_split")
            if locked_split is not None:
                negative_split.value = locked_split
                negative_split.disabled = True
                negative_split_info.value = (
                    f"This recording is locked to {locked_split}; all its samples use that split."
                )
            else:
                negative_split.disabled = recording
                negative_split_info.value = "Choose a split before the first save; it will then be locked for this recording."
            intervals = negative_extraction_intervals()
            if not intervals:
                raise ValueError(
                    "Add at least one interval in Advanced, or turn off several intervals to use the slider."
                )
            frames = []
            for start, end in sorted(intervals):
                if not (
                    np.isfinite(start) and np.isfinite(end) and 0 <= start < end <= info.duration
                ):
                    raise ValueError("Choose a nonempty section inside the recording.")
                first = int(np.ceil(start * info.samplerate))
                last = info.frames if end == info.duration else int(np.floor(end * info.samplerate))
                if last <= first or round((last - first) * SAMPLE_RATE / info.samplerate) < 1:
                    raise ValueError("Choose a section containing at least one audio sample.")
                if frames and first < frames[-1][1]:
                    raise ValueError("Time intervals overlap. Remove the overlap in Advanced.")
                frames.append((first, last))
            target = round(round(window.value * SAMPLE_RATE) * info.samplerate / SAMPLE_RATE)
            sizes = [b - a for a, b in frames]
            duration = sum(sizes) / info.samplerate
            prefix = f"Selected: {duration:.2f}s. Each saved sample: {window.value:.2f}s. "
            if method in {"clips", "placement"}:
                if any(size > target for size in sizes):
                    raise ValueError(prefix + "Select a shorter clip or increase Output (s).")
                padding = sum(max(0, target - size) for size in sizes) / info.samplerate
                if method == "placement":
                    total = len(sizes) * negative_count.value
                    message = (
                        prefix
                        + f"Keep each complete clip. Save {negative_count.value} random placement(s) per clip ({total} total). Fill the rest with silence."
                    )
                    if all(size == target for size in sizes):
                        message += " The clip fills the output window, so its position cannot move."
                else:
                    total = len(sizes)
                    message = (
                        prefix
                        + f"Save {total} centered sample(s); add {padding:.2f}s of silence in total."
                    )
                extract_button.description = (
                    "Save 1 sample" if total == 1 else f"Save {total} samples"
                )
            else:
                if any(size < target for size in sizes):
                    raise ValueError(prefix + "Select a longer section or reduce Output (s).")
                if method == "random":
                    capacity = sum(size - target + 1 for size in sizes)
                    negative_count.max = min(10000, capacity)
                    message = (
                        prefix
                        + f"Save {negative_count.value} random sample(s) inside the selected section(s). Overlap is allowed."
                    )
                else:
                    stride = negative_stride.value
                    if (
                        not np.isfinite(stride)
                        or stride <= 0
                        or round(stride * info.samplerate) < 1
                    ):
                        raise ValueError("Set a positive Stride in seconds.")
                    total = sum(
                        (size - target) // round(stride * info.samplerate) + 1 for size in sizes
                    )
                    message = prefix + f"Save {total} sample(s), moving {stride:.2f}s each time."
                extract_button.description = "Save samples"
            negative_summary.value = message
            extract_button.disabled = recording
        except ValueError as error:
            negative_summary.value = str(error)
            extract_button.disabled = True
    finally:
        negative_ui_state["updating"] = False


def toggle_multiple_intervals(change):
    if negative_ui_state["updating"]:
        return
    if change["new"] and not negative_ranges.value.strip() and negative_source.value is not None:
        start, end = negative_selection.value
        negative_ranges.value = f"{start!r},{end!r}"
    update_negative_preparation()


def sync_output_window(change):
    other = negative_window if change["owner"] is window else window
    if other.value != change["new"]:
        other.value = change["new"]
    update_negative_preparation()


def selected_negative_range():
    if negative_source.value is None:
        raise ValueError("Select a negative source first.")
    path = Path(negative_source.value)
    info = sf.info(path)
    start, end = negative_selection.value
    if not (np.isfinite(start) and np.isfinite(end) and 0 <= start < end <= info.duration):
        raise ValueError("Select a nonempty interval inside the recording.")
    first = int(np.ceil(start * info.samplerate))
    last = info.frames if end == info.duration else int(np.floor(end * info.samplerate))
    if last <= first or round((last - first) * SAMPLE_RATE / info.samplerate) < 1:
        raise ValueError("Select an interval containing at least one audio sample.")
    return path, info, start, end, first, last


def preview_negative_source():
    path, info, start, end, first, last = selected_negative_range()
    audio = read_normalized_segment(path, first, last - first)
    show_audio(audio, f"Selected source interval: {start:.3f}-{end:.3f}s")


def use_negative_range(append=False):
    path, info, start, end, first, last = selected_negative_range()
    intervals = parse_negative_ranges() if append else []
    intervals.append((start, end))
    intervals.sort()
    if any(
        not (np.isfinite(a) and np.isfinite(b) and 0 <= a < b <= info.duration)
        for a, b in intervals
    ):
        raise ValueError("Every range must be a nonempty interval inside this source.")
    if any(current[0] < previous[1] for previous, current in zip(intervals, intervals[1:])):
        raise ValueError(
            "Ranges overlap. Use range to replace the list, or select a separate interval."
        )
    negative_ranges.value = "\n".join(f"{a!r},{b!r}" for a, b in intervals)
    negative_multiple.value = True
    log("Interval list updated:", len(intervals), "interval(s).")
    update_negative_preparation()


def import_old():
    imported = import_legacy_recordings()
    refresh_sources(state["source"])
    log("Imported", len(imported), "recordings. Old files were preserved.")


def on_source_change(change):
    if change["new"] is not None and not state["updating_picker"]:
        run_action(lambda: load_positive(change["new"]))


def action_row(controls, explanation):
    return widgets.HBox(
        [
            *controls,
            widgets.HTML(
                value=f"<span style='color:var(--vscode-descriptionForeground, #666)'>{explanation}</span>",
                layout={"flex": "1 1 320px"},
            ),
        ],
        layout={"align_items": "center", "flex_flow": "row wrap"},
    )


negative_ranges_box = widgets.VBox(
    [
        widgets.HTML(
            value="With several intervals enabled, this list replaces the slider for saving samples."
        ),
        negative_ranges,
        action_row(
            [negative_use_range_button], "Replace the interval list with the slider selection."
        ),
        action_row(
            [negative_add_range_button], "Append the slider selection to the interval list."
        ),
    ]
)
negative_advanced = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                negative_split,
                negative_split_info,
                negative_multiple,
                negative_ranges_box,
            ]
        )
    ],
    selected_index=None,
)
negative_advanced.set_title(0, "Advanced: dataset split and several intervals")
legacy_tools = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                action_row(
                    [legacy_button],
                    "Copy recordings from the old pilot folders into the current layout.",
                )
            ]
        )
    ],
    selected_index=None,
)
legacy_tools.set_title(0, "Legacy data tools")


refresh_sources()
source_picker.observe(on_source_change, names="value")
refresh_negative_sources()
refresh_processed_samples()
negative_kind.observe(change_negative_category, names="value")
negative_source.observe(
    lambda change: (
        run_action(load_negative_source) if not negative_picker_state["updating"] else None
    ),
    names="value",
)
for control in [
    negative_selection,
    negative_mode,
    negative_count,
    negative_stride,
    negative_ranges,
    negative_split,
]:
    control.observe(update_negative_preparation, names="value")
negative_multiple.observe(toggle_multiple_intervals, names="value")
window.observe(sync_output_window, names="value")
negative_window.observe(sync_output_window, names="value")
extract_button.on_click(lambda _: run_action(extract_selected_negatives))
negative_preview_button.on_click(lambda _: run_action(preview_negative_source))
negative_use_range_button.on_click(lambda _: run_action(use_negative_range))
negative_add_range_button.on_click(lambda _: run_action(lambda: use_negative_range(append=True)))
hard_negative_button.on_click(lambda _: run_action(lambda: start_recording("hard_negative")))
hard_negative_stop_button.on_click(lambda _: run_action(stop_recording))
record_button.on_click(lambda _: run_action(lambda: start_recording("positive")))
record_stop_button.on_click(lambda _: run_action(stop_recording))
negative_button.on_click(lambda _: run_action(lambda: start_recording("negative")))
negative_stop_button.on_click(lambda _: run_action(stop_recording))
noise_button.on_click(lambda _: run_action(lambda: start_recording("noise")))
noise_stop_button.on_click(lambda _: run_action(stop_recording))
preview_button.on_click(
    lambda _: run_action(lambda: show_audio(selected_phrase(), "Selected phrase"))
)
augment_button.on_click(lambda _: run_action(generate_variants))
batch_button.on_click(lambda _: run_action(lambda: augment_all("positive")))
negative_aug_button.on_click(lambda _: run_action(augment_chosen_negatives))
negative_processed_refresh.on_click(lambda _: run_action(refresh_processed_samples))
negative_processed.observe(update_processed_augmentation, names="value")
negative_aug_scope.observe(update_processed_augmentation, names="value")
legacy_button.on_click(lambda _: run_action(import_old))
_pilot_panel = widgets.VBox(
    [
        device,
        window,
        source_picker,
        selection,
        action_row(
            [record_button, record_stop_button],
            "Record a positive example. Stop saves raw and prepares the selected positive source.",
        ),
        action_row(
            [negative_button, negative_stop_button],
            "Record ordinary negatives. Stop saves raw; use Save samples below to process it.",
        ),
        action_row(
            [hard_negative_button, hard_negative_stop_button],
            "Record a hard-negative example. Stop saves raw; use Save samples below to process it.",
        ),
        action_row(
            [noise_button, noise_stop_button],
            "Record background noise into noise/raw/ for mixing into augmented samples.",
        ),
        recording_status,
        action_row([preview_button], "Listen to the selected positive phrase."),
        action_row(
            [augment_button],
            "Save the selected phrase and create 6 augmented variants for this positive source.",
        ),
        action_row(
            [batch_button],
            "Create variants for ALL positive raw recordings, using their saved phrase selections.",
        ),
        widgets.HTML(value="<b>Prepare negative samples</b>"),
        negative_kind,
        negative_source,
        negative_info,
        negative_selection,
        action_row(
            [negative_preview_button],
            "Listen to the current slider selection. Save samples creates the processed audio.",
        ),
        negative_window,
        negative_mode,
        negative_count,
        negative_stride,
        negative_summary,
        negative_advanced,
        action_row(
            [extract_button],
            "Selected Source + slider interval (or the enabled Advanced interval list) → processed/. Each click saves a new set.",
        ),
        widgets.HTML(value="<b>Augment processed audio</b>"),
        negative_aug_scope,
        negative_processed,
        action_row(
            [negative_processed_refresh],
            "File labels include the source and extraction folder. Ctrl/Cmd-click for several files; Shift-click for a range.",
        ),
        widgets.HBox(
            [negative_aug_button, negative_aug_summary],
            layout={"align_items": "center", "flex_flow": "row wrap"},
        ),
        legacy_tools,
        output,
    ]
)
display(_pilot_panel)
update_controls()
log("Ready. Recording starts only when you click a recording button.")
log("Noise is shared across sessions from data/noise/raw/.")

## What to check

- Listen to the complete positive recording or your manual selection. The notebook does not detect or recognize "Hey GPT". All augmented copies inherit any labeling or selection mistake.
- Positive variants include three random placements, one quieter sample, one noise sample, and one faster sample. Ordinary and hard-negative windows receive gain, noise, and speed variants; both retain label 0. A mixed positive remains positive; a mixed negative remains negative. Noise sources do not automatically become negative training examples.
- Window length is the length of each augmented example, independent of the Start/Stop recording duration. Oversized positives are rejected or skipped by batch augmentation with a message.
- Raw samples are preserved. Processed and augmented WAVs use mono 16 kHz PCM16. Clipped microphone input cannot be repaired by later processing.
- Split speakers and original sources before building a train/validation/test dataset. All descendants of a source must stay in its split; the shared pilot noise pool is not split-aware.
- Stop any active recording before rerunning a cell or restarting the kernel. Rerun guards prevent replacing a live recorder and losing its Stop control.
- **Import old recordings** copies older pilot sources without deleting them or duplicating already imported sources.

Clear notebook outputs before committing; recordings and generated audio are ignored by Git.

## Preparing Ordinary and Hard-Negative Samples

1. Choose **Category** and **Source**, then drag **Selection (s)** and click **Preview selection**. The entire file is initially selected; the slider interval is used directly when saving.
2. Set **Output (s)** for each saved sample's duration. It is linked to Window above. The live summary shows the selection length, output length, and padding/sampling behavior.
3. Hard negatives default to **Random position in output window**. The selected clip is normalized once, kept intact, and placed at a random offset; silence fills the rest. Set **Copies / clip** to 1 for one placement or a larger number for more independently placed copies. With several intervals, the count applies to each clip. Positions may repeat; an exact-fit clip cannot move. **Centered in output window** makes one centered copy per clip. Oversized clips are rejected. Ordinary negatives instead use **Random crops from recording** with **Samples**, or **Slide through selection** with **Stride (s)**. The hard-negative UI never uses the crop methods. The save button explains invalid selections before submission.
4. Click **Save samples** (or **Save 1 sample**). The newly saved processed files are selected automatically under **Augment processed audio**, replacing the previous selection. Keep **Selected processed files** and select one or more files (Ctrl/Cmd-click for individual files, Shift-click for a range), then click **Augment N selected**. To include every processed file in Category across all raw sources, explicitly choose **All processed files in Category** and click **Augment all N files**. The file list shows source/extraction folders; **Refresh file list** reloads existing files and preserves valid selections. Augmentation applies gain/noise/speed variants, preserves each saved sample's length, and creates a new run on each click. Changing Category clears the file selection and returns to selected-file mode.
5. Optional: open **Advanced: dataset split and several intervals**. Split defaults to unassigned and is locked for that source after the first save. Group different recordings by speaker/session yourself before assigning splits.
6. Enable **Use several time intervals** only when you need disjoint ranges. The list then replaces the slider for saving. Use **Replace interval list** or **Add selection to list**, or edit `start,end` lines manually. Disable this checkbox to return to direct slider selection. Overlapping intervals are rejected.

Stop saves raw without automatic preview or processing. Source selection reads metadata only; preview decodes only the selected interval, so memory use grows with its duration. Sampling and extraction seek/read one window at a time. No window crosses an approved interval. Each save creates a new folder with source frame bounds, padding, mode, seed, split, and source group. Both negative categories retain label 0.


## Standalone control panel - run this cell only

For teammates: use the project environment and run only the next cell. It includes all setup, helpers, and controls; the three earlier cells remain available for step-by-step use.

Choose **Start positive**, **Start negative**, **Start hard negative**, or **Start noise**, then the matching **Stop** button. There is no countdown, fixed recording duration, or automatic wake-word cropping. Use the full positive recording or adjust Phrase manually before augmentation.

For negative categories, choose a source and slider interval in **Prepare negative samples**, then **Save samples**. In **Augment processed audio**, augment the selected processed files or explicitly choose **All processed files in Category**. Newly saved files are selected automatically. Split and optional multiple intervals are under Advanced. Stop saves raw without processing the full recording.


In [ ]:
import asyncio
import json
import random
import shutil
from bisect import bisect_right
from datetime import datetime, timezone
from math import gcd
from pathlib import Path
from threading import Event, Thread
from uuid import uuid4

import ipywidgets as widgets
import numpy as np
import sounddevice as sd
import soundfile as sf
from audiomentations import AddBackgroundNoise, AddGaussianSNR, Gain, TimeStretch
from IPython.display import SVG, Audio, display
from scipy.signal import resample_poly

previous_recorder = globals().get("_recorder") or globals().get("_noise_recorder")
if previous_recorder is not None and previous_recorder.running:
    raise RuntimeError("Stop recording before rerunning setup or the standalone cell.")
previous_status_task = globals().get("_recording_status_task")
if previous_status_task is not None:
    previous_status_task.cancel()

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the GPTWake project.")

DATA_DIR = ROOT / "data"
SAMPLE_RATE = 16000
SPEAKER_ID = "self"
state = {"audio": None, "source": None, "updating_picker": False}
print("Dataset root:", DATA_DIR)
print(
    "Layout: positive, negative, and hard_negative have raw/processed/augmented; noise uses raw only."
)
print("No folders or recordings are created until you save data.")


def log(*values):
    target = globals().get("output")
    if target is None:
        print(*values)
    else:
        target.append_stdout(" ".join(str(value) for value in values) + "\n")


def emit_preview(value):
    # Update widget state directly; do not also publish a captured IOPub display.
    target = globals().get("output")
    if target is None:
        display(value)
    else:
        target.append_display_data(value)


def close_panel(widget):
    for child in getattr(widget, "children", ()):
        close_panel(child)
    if hasattr(widget, "disabled"):
        widget.disabled = True
    widget.close()


def dataset_path(kind, stage):
    if kind not in {"positive", "negative", "hard_negative", "noise"} or stage not in {
        "raw",
        "processed",
        "augmented",
    }:
        raise ValueError("Unknown dataset category or stage.")
    return DATA_DIR / kind / stage


def next_path(directory, prefix, suffix=".wav"):
    # Inspect existing names on disk so restarting the kernel cannot overwrite a recording.
    number = 1
    for path in directory.glob(f"{prefix}_*"):
        tail = path.stem.rsplit("_", 1)[-1]
        if tail.isdigit():
            number = max(number, int(tail) + 1)
    return directory / f"{prefix}_{number:04d}{suffix}"


def write_metadata(path, metadata):
    path.write_text(
        json.dumps(
            metadata, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else str(x)
        ),
        encoding="utf-8",
    )


def write_wav(path, audio, sample_rate=SAMPLE_RATE, subtype="PCM_16", overwrite=True):
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim not in {1, 2} or not len(audio) or not np.isfinite(audio).all():
        raise ValueError("Expected nonempty, finite audio.")
    if sample_rate <= 0:
        raise ValueError("Sample rate must be positive.")
    peak = float(np.max(np.abs(audio)))
    if subtype != "FLOAT" and peak > 1:
        log(f"Reducing peak {peak:.3f} to avoid PCM clipping: {path.name}")
        audio = audio * (0.999 / peak)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb" if overwrite else "xb") as stream:
        sf.write(stream, audio, sample_rate, format="WAV", subtype=subtype)
    return audio


def normalize_source(path):
    audio, rate = sf.read(path, dtype="float32", always_2d=True)
    audio = audio.mean(axis=1)
    if rate != SAMPLE_RATE:
        divisor = gcd(rate, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // divisor, rate // divisor)
    return np.asarray(audio, dtype=np.float32)


def save_recording(kind, audio, sample_rate, metadata=None):
    path = next_path(dataset_path(kind, "raw"), kind)
    write_wav(path, audio, sample_rate, subtype="FLOAT", overwrite=False)
    write_metadata(
        path.with_suffix(".json"),
        {
            **(metadata or {}),
            "recording_id": path.stem,
            "category": kind,
            "label": {"positive": 1, "negative": 0, "hard_negative": 0, "noise": None}[kind],
            "speaker_id": SPEAKER_ID,
            "split": "unassigned",
            "raw_sample_rate": sample_rate,
            "created_at": datetime.now(timezone.utc).isoformat(),
        },
    )
    return path


def process_recording(kind, source, bounds=None, window_seconds=2.0):
    source = Path(source)
    if kind == "noise":
        return [source]
    sidecar = source.with_suffix(".json")
    source_metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
    if source_metadata.get("status", "complete") != "complete":
        raise ValueError("Only completed recordings can be processed.")
    if kind in {"negative", "hard_negative"}:
        raise ValueError(
            "Use the negative extraction panel to choose approved time intervals first."
        )
    audio = normalize_source(source)
    common = {
        "source": str(source.relative_to(ROOT)),
        "recording_id": source.stem,
        "category": kind,
        "label": {"positive": 1, "negative": 0, "hard_negative": 0, "noise": None}[kind],
        "sample_rate": SAMPLE_RATE,
        "speaker_id": source_metadata.get("speaker_id", SPEAKER_ID),
        "split": source_metadata.get("split", "unassigned"),
    }
    directory = dataset_path(kind, "processed")
    if kind == "positive":
        selection_method = "manual" if bounds is not None else "full_recording"
        if bounds is None:
            old_meta = directory / f"{source.stem}.json"
            saved = json.loads(old_meta.read_text()) if old_meta.exists() else {}
            # Legacy intervals may have been selected by the removed energy heuristic.
            if saved.get("selection_method") == "manual":
                bounds = saved["selection_seconds"]
                selection_method = "manual"
            else:
                bounds = (0.0, len(audio) / SAMPLE_RATE)
        common["selection_method"] = selection_method
        start, end = map(float, bounds)
        if not 0 <= start < end <= len(audio) / SAMPLE_RATE:
            raise ValueError("Phrase bounds must lie within the original recording.")
        segments = [
            (
                source.stem,
                audio[round(start * SAMPLE_RATE) : round(end * SAMPLE_RATE)],
                [start, end],
            )
        ]
    outputs = []
    for stem, segment, interval in segments:
        path = directory / f"{stem}.wav"
        write_wav(path, segment)
        write_metadata(path.with_suffix(".json"), {**common, "selection_seconds": interval})
        outputs.append(path)
    return outputs


def read_normalized_segment(source, start_frame, frame_count):
    with sf.SoundFile(source) as stream:
        rate = stream.samplerate
        stream.seek(start_frame)
        audio = stream.read(frame_count, dtype="float32", always_2d=True)
    if len(audio) != frame_count or not len(audio) or not np.isfinite(audio).all():
        raise ValueError("The selected segment is empty, incomplete, or contains invalid samples.")
    audio = audio.mean(axis=1)
    if rate != SAMPLE_RATE:
        divisor = gcd(rate, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // divisor, rate // divisor)
    expected = round(frame_count * SAMPLE_RATE / rate)
    return np.asarray(audio[:expected], dtype=np.float32)


def extract_negative_windows(
    kind,
    source,
    intervals,
    window_seconds=2.0,
    mode="random",
    count=20,
    stride_seconds=None,
    split=None,
    seed=None,
):
    if kind not in {"negative", "hard_negative"}:
        raise ValueError("Choose negative or hard_negative for window extraction.")
    source = Path(source)
    if source.resolve().parent != dataset_path(kind, "raw").resolve():
        raise ValueError("The selected source must belong to this category's raw directory.")
    sidecar = source.with_suffix(".json")
    metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
    if metadata.get("status", "complete") != "complete":
        raise ValueError("Only completed recordings can be processed.")
    info = sf.info(source)
    if not np.isfinite(window_seconds) or window_seconds <= 0:
        raise ValueError("Window duration must be positive and finite.")
    window_size = round(window_seconds * SAMPLE_RATE)
    native_size = round(window_size * info.samplerate / SAMPLE_RATE)
    if window_size < 1 or native_size < 1 or info.frames < 1:
        raise ValueError("Source and window must contain audio samples.")
    if mode not in {"random", "sliding", "clips", "placement"}:
        raise ValueError("Unknown window extraction mode.")
    if mode in {"clips", "placement"} and kind != "hard_negative":
        raise ValueError("Clip padding is intended for separately recorded hard negatives.")
    if not intervals:
        raise ValueError("Enter at least one approved start,end interval in seconds.")
    ranges = []
    for interval in intervals:
        if len(interval) != 2:
            raise ValueError("Each interval needs a start and end in seconds.")
        start, end = map(float, interval)
        if not (np.isfinite(start) and np.isfinite(end) and 0 <= start < end <= info.duration):
            raise ValueError("Each interval must lie within the source duration.")
        # Quantize inward: a window must never read outside an approved interval.
        first = int(np.ceil(start * info.samplerate))
        last = info.frames if end == info.duration else int(np.floor(end * info.samplerate))
        if last <= first or round((last - first) * SAMPLE_RATE / info.samplerate) < 1:
            raise ValueError(
                "An interval is shorter than one audio sample at the source or target rate."
            )
        if mode in {"clips", "placement"} and last - first > native_size:
            raise ValueError(
                "Hard-negative clip exceeds the window; trim manually or enlarge the window."
            )
        if mode not in {"clips", "placement"} and last - first < native_size:
            raise ValueError("Every interval must contain at least one complete window.")
        ranges.append((first, last))
    ranges.sort()
    if any(current[0] < previous[1] for previous, current in zip(ranges, ranges[1:])):
        raise ValueError("Approved intervals must not overlap.")
    split = metadata.get("split", "unassigned") if split is None else split
    if split not in {"unassigned", "train", "validation", "test"}:
        raise ValueError("Unknown dataset split.")
    if "window_split" in metadata and metadata["window_split"] != split:
        raise ValueError(
            f"This source is locked to split {metadata['window_split']}; keep the same split."
        )
    seed = uuid4().int % (2**32) if seed is None else int(seed)
    if mode in {"random", "placement"}:
        if not isinstance(count, (int, np.integer)) or isinstance(count, bool) or count < 1:
            raise ValueError("Sample count must be a positive integer.")
    if mode == "random":
        lengths = [last - first - native_size + 1 for first, last in ranges]
        cumulative = np.cumsum(lengths).tolist()
        if count > cumulative[-1]:
            raise ValueError("Requested count exceeds the number of unique window positions.")
        # Sampling an integer range avoids allocating an array for an hours-long track.
        positions = random.Random(seed).sample(range(cumulative[-1]), count)
        selected = []
        for position in positions:
            index = bisect_right(cumulative, position)
            offset = position - (cumulative[index - 1] if index else 0)
            start = ranges[index][0] + offset
            selected.append((start, start + native_size))
    elif mode == "sliding":
        stride_seconds = window_seconds if stride_seconds is None else stride_seconds
        if not np.isfinite(stride_seconds) or stride_seconds <= 0:
            raise ValueError("Sliding stride must be positive and finite.")
        stride = round(stride_seconds * info.samplerate)
        if stride < 1:
            raise ValueError("Sliding stride must be at least one source sample.")
        selected = (
            (start, start + native_size)
            for first, last in ranges
            for start in range(first, last - native_size + 1, stride)
        )
    else:
        selected = iter(ranges)
    relative = str(source.relative_to(ROOT))
    folder = next_path(dataset_path(kind, "processed"), source.stem + "_extract", suffix="")
    source_group = metadata.get("source_group", relative)
    metadata.update(split=split, window_split=split, source_group=source_group)
    write_metadata(sidecar, metadata)
    outputs = []
    placement_rng = np.random.default_rng(seed) if mode == "placement" else None
    for start, end in selected:
        audio = read_normalized_segment(source, start, end - start)
        if not len(audio):
            raise ValueError("Selected clip is shorter than one target sample.")
        if mode in {"clips", "placement"} and len(audio) > window_size:
            raise ValueError(
                "The normalized clip exceeds the output window; no audio was truncated."
            )
        # Complete-window crops may differ by one sample due to resampling rounding.
        audio = audio[:window_size]
        for copy_index in range(count if mode == "placement" else 1):
            slack = window_size - len(audio)
            if mode == "placement":
                left = int(placement_rng.integers(0, slack + 1))
            else:
                left = slack // 2 if mode == "clips" else 0
            right = slack - left
            placed = np.pad(audio, (left, right))
            path = folder / f"window_{len(outputs) + 1:04d}.wav"
            write_wav(path, placed, overwrite=False)
            write_metadata(
                path.with_suffix(".json"),
                {
                    "source": relative,
                    "recording_id": source.stem,
                    "category": kind,
                    "label": 0,
                    "sample_rate": SAMPLE_RATE,
                    "speaker_id": metadata.get("speaker_id", SPEAKER_ID),
                    "split": split,
                    "source_group": source_group,
                    "selection_seconds": [start / info.samplerate, end / info.samplerate],
                    "source_start_frame": start,
                    "source_end_frame": end,
                    "source_sample_rate": info.samplerate,
                    "window_seconds": window_size / SAMPLE_RATE,
                    "windowing_mode": mode,
                    "sampling_seed": seed if mode in {"random", "placement"} else None,
                    "placement_offset_samples": left if mode == "placement" else None,
                    "placement_offset_seconds": left / SAMPLE_RATE if mode == "placement" else None,
                    "padding_left_samples": left,
                    "padding_right_samples": right,
                },
            )
            outputs.append(path)
    write_metadata(
        folder / "extraction.json",
        {
            "source": relative,
            "source_group": source_group,
            "category": kind,
            "label": 0,
            "split": split,
            "mode": mode,
            "window_seconds": window_size / SAMPLE_RATE,
            "approved_intervals_seconds": [
                [a / info.samplerate, b / info.samplerate] for a, b in ranges
            ],
            "seed": seed if mode in {"random", "placement"} else None,
            "copies_per_clip": count if mode == "placement" else None,
            "stride_seconds": stride / info.samplerate if mode == "sliding" else None,
            "window_count": len(outputs),
        },
    )
    log("Saved", len(outputs), kind, "windows:", folder.relative_to(ROOT))
    return outputs


def processed_negative_windows(kind):
    if kind not in {"negative", "hard_negative"}:
        raise ValueError("Choose a negative category.")
    outputs = []
    for path in sorted(dataset_path(kind, "processed").rglob("*.wav")):
        sidecar = path.with_suffix(".json")
        if not sidecar.exists():
            continue
        metadata = json.loads(sidecar.read_text())
        if metadata.get("category") == kind and metadata.get("label") == 0:
            outputs.append(path)
    return outputs


def prepare_noise_library():
    # Load only the segment needed for mixing; long tracks are never normalized in full.
    return dataset_path("noise", "raw")


def available_audio_files(directory):
    candidates = []
    for path in sorted(directory.glob("*.wav")):
        sidecar = path.with_suffix(".json")
        try:
            metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
            if metadata.get("status", "complete") != "complete":
                continue
            if sf.info(path).frames > 0:
                candidates.append(path)
        except (OSError, ValueError, RuntimeError):
            continue
    return candidates


def make_noise_transform(noise_dir):
    candidates = available_audio_files(noise_dir)
    if candidates:
        return AddBackgroundNoise(sounds_path=candidates, min_snr_db=10, max_snr_db=25, p=1)
    return AddGaussianSNR(min_snr_db=10, max_snr_db=25, p=1)


def augment_recording(kind, processed_source, window_seconds=2.0, show_players=False):
    if kind not in {"positive", "negative", "hard_negative"}:
        raise ValueError("Only positive, negative, and hard_negative examples are augmented here.")
    processed_source = Path(processed_source)
    audio, rate = sf.read(processed_source, dtype="float32")
    if rate != SAMPLE_RATE or audio.ndim != 1:
        raise ValueError("Process the source into mono 16 kHz audio first.")
    metadata = json.loads(processed_source.with_suffix(".json").read_text())
    seed = uuid4().int % (2**32)
    random.seed(seed)
    noise_dir = prepare_noise_library()
    if kind == "positive":
        variants = make_variants(audio, window_seconds, noise_dir, seed)
    else:
        # Negative windows already have fixed length; avoid three identical placement copies.
        quiet = Gain(min_gain_db=-9, max_gain_db=-3, p=1)
        stretch = TimeStretch(min_rate=1.03, max_rate=1.10, leave_length_unchanged=True, p=1)
        noise = make_noise_transform(noise_dir)
        variants = []
        for name, transform in [("quieter", quiet), ("noise", noise), ("faster", stretch)]:
            result = transform(samples=audio.copy(), sample_rate=SAMPLE_RATE)
            variants.append((name, result, dict(transform.parameters)))
    prefix = processed_source.stem
    if kind in {"negative", "hard_negative"} and processed_source.parent != dataset_path(
        kind, "processed"
    ):
        prefix = processed_source.parent.name + "_" + prefix
    folder = next_path(dataset_path(kind, "augmented"), prefix + "_run", suffix="")
    entries = []
    for name, result, parameters in variants:
        result = write_wav(folder / f"{name}.wav", result, overwrite=False)
        parameters = dict(parameters)
        if "noise_file_path" in parameters:
            parameters["noise_file_path"] = str(
                Path(parameters["noise_file_path"]).relative_to(ROOT)
            )
        entries.append({"file": f"{name}.wav", "parameters": parameters})
        if show_players:
            show_audio(result, name)
    write_metadata(
        folder / "metadata.json",
        {
            **metadata,
            "processed_source": str(processed_source.relative_to(ROOT)),
            "window_seconds": len(variants[0][1]) / SAMPLE_RATE,
            "augmentation_seed": seed,
            "variants": entries,
        },
    )
    log("Saved", len(variants), "variants:", folder.relative_to(ROOT))
    return folder


def import_legacy_recordings():
    # Copy old sources without deleting or re-encoding them. Repeated imports are safe.
    imported = []
    patterns = {
        "positive": "raw/positive_pilot/**/*.wav",
        "noise": "negatives/room_noise_pilot/**/*.wav",
    }
    for kind, pattern in patterns.items():
        known = {
            json.loads(path.read_text()).get("legacy_source")
            for path in dataset_path(kind, "raw").glob("*.json")
        }
        for source in sorted(DATA_DIR.glob(pattern)):
            relative = str(source.relative_to(ROOT))
            if relative in known:
                continue
            destination = next_path(dataset_path(kind, "raw"), kind)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with source.open("rb") as src, destination.open("xb") as dst:
                shutil.copyfileobj(src, dst)
            old_sidecar = source.with_suffix(".json")
            write_metadata(
                destination.with_suffix(".json"),
                {
                    "recording_id": destination.stem,
                    "category": kind,
                    "speaker_id": SPEAKER_ID,
                    "split": "unassigned",
                    "legacy_source": relative,
                    "raw_sample_rate": sf.info(destination).samplerate,
                    "original_metadata": json.loads(old_sidecar.read_text())
                    if old_sidecar.exists()
                    else {},
                },
            )
            process_recording(kind, destination)
            known.add(relative)
            imported.append(destination)
    return imported


def show_audio(audio, title):
    peak = float(np.max(np.abs(audio)))
    log(f"{title}: {len(audio) / SAMPLE_RATE:.2f}s, peak={peak:.3f}")
    playback = audio
    if peak > 1:
        # Float capture and resampling can exceed the player's [-1, 1] range.
        playback = audio * (0.999 / peak)
        log("Preview volume reduced to fit playback range; source audio is unchanged.")
    emit_preview(Audio(playback, rate=SAMPLE_RATE, normalize=False))
    # Display every bin's peak so short transients remain visible.
    bins = np.array_split(np.abs(audio), min(500, len(audio)))
    peaks = [float(b.max()) for b in bins]
    lines = "".join(
        f'<line x1="{i * 1.2:.1f}" x2="{i * 1.2:.1f}" '
        f'y1="{45 - v * 42:.1f}" y2="{45 + v * 42:.1f}" '
        'stroke="#2475c0"/>'
        for i, v in enumerate(peaks)
    )
    emit_preview(
        SVG(
            f'<svg xmlns="http://www.w3.org/2000/svg" width="600" height="90" '
            f'viewBox="0 0 600 90"><rect width="600" height="90" fill="#f3f6fa"/>{lines}</svg>'
        )
    )


def random_place(phrase, window_seconds, rng):
    total = round(window_seconds * SAMPLE_RATE)
    if not 0 < len(phrase) <= total:
        raise ValueError(
            "The selected phrase does not fit. Adjust its bounds or increase the window."
        )
    offset = int(rng.integers(0, total - len(phrase) + 1))
    result = np.zeros(total, dtype=np.float32)
    result[offset : offset + len(phrase)] = phrase
    return result, offset / SAMPLE_RATE


def make_variants(phrase, window_seconds, noise_dir, seed):
    rng = np.random.default_rng(seed)
    if len(phrase) > round(window_seconds * SAMPLE_RATE):
        raise ValueError(
            "Phrase exceeds the window; no positive was truncated. Adjust selection or window."
        )
    variants = []
    for i in range(3):
        audio, offset = random_place(phrase, window_seconds, rng)
        variants.append((f"placement_{i + 1}", audio, {"offset_seconds": offset}))
    quiet = Gain(min_gain_db=-9, max_gain_db=-3, p=1)
    quieter = quiet(samples=phrase.copy(), sample_rate=SAMPLE_RATE)
    audio, offset = random_place(quieter, window_seconds, rng)
    variants.append(("quieter", audio, {"offset_seconds": offset, **quiet.parameters}))
    audio, offset = random_place(phrase, window_seconds, rng)
    noise = make_noise_transform(noise_dir)
    label = "room_noise" if isinstance(noise, AddBackgroundNoise) else "gaussian_noise_demo"
    noisy = noise(samples=audio.copy(), sample_rate=SAMPLE_RATE)
    variants.append((label, noisy, {"offset_seconds": offset, **noise.parameters}))
    # Faster speech fits whenever the original phrase fits. Preserve pitch and do not pad/crop it here.
    stretch = TimeStretch(min_rate=1.03, max_rate=1.10, leave_length_unchanged=False, p=1)
    faster = stretch(samples=phrase.copy(), sample_rate=SAMPLE_RATE)
    audio, offset = random_place(faster, window_seconds, rng)
    variants.append(("faster", audio, {"offset_seconds": offset, **stretch.parameters}))
    return variants


class AudioRecorder:
    """Capture native-rate blocks to disk while the notebook remains interactive."""

    def __init__(self):
        self.path = None
        self.kind = None
        self.frames = 0
        self.rate = 0
        self.error = None
        self.overflows = 0
        self.ready = Event()
        self._stop = Event()
        self._thread = None
        self._stream = None
        self._metadata = {}

    @property
    def running(self):
        return self._thread is not None and self._thread.is_alive()

    def start(self, device, kind="noise"):
        if self.running:
            raise RuntimeError("A recording is already running.")
        if kind not in {"positive", "negative", "hard_negative", "noise"}:
            raise ValueError("Unknown recording category.")
        info = sd.query_devices(device, "input")
        rate = int(info["default_samplerate"])
        sd.check_input_settings(device=device, channels=1, samplerate=rate, dtype="float32")
        if not sf.check_format("RF64", "FLOAT"):
            raise RuntimeError("This libsndfile installation does not support RF64 float WAV.")
        path = next_path(dataset_path(kind, "raw"), kind)
        path.parent.mkdir(parents=True, exist_ok=True)
        handle = path.open("xb")
        self.path, self.rate, self.kind = path, rate, kind
        self.frames, self.overflows, self.error = 0, 0, None
        self.ready.clear()
        self._stop.clear()
        self._metadata = {
            "recording_id": path.stem,
            "category": kind,
            "label": {"positive": 1, "negative": 0, "hard_negative": 0, "noise": None}[kind],
            "speaker_id": SPEAKER_ID,
            "split": "unassigned",
            "device": str(info["name"]),
            "raw_sample_rate": rate,
            "capture_rate": rate,
            "format": "RF64",
            "subtype": "FLOAT",
            "created_at": datetime.now(timezone.utc).isoformat(),
            "status": "recording",
        }
        try:
            write_metadata(path.with_suffix(".json"), self._metadata)
            self._thread = Thread(target=self._record, args=(handle, device), daemon=True)
            self._thread.start()
        except Exception:
            handle.close()
            raise
        return path

    def _record(self, handle, device):
        try:
            # RF64 keeps the .wav extension and can exceed the ordinary WAV 4 GiB limit.
            with handle:
                with sf.SoundFile(
                    handle,
                    mode="w",
                    samplerate=self.rate,
                    channels=1,
                    format="RF64",
                    subtype="FLOAT",
                ) as sink:
                    with sd.InputStream(
                        device=device,
                        samplerate=self.rate,
                        channels=1,
                        dtype="float32",
                        blocksize=4096,
                    ) as stream:
                        self._stream = stream
                        self.ready.set()
                        while not self._stop.is_set():
                            block, overflowed = stream.read(4096)
                            if len(block):
                                sink.write(block)
                                sink.flush()
                                self.frames += len(block)
                            self.overflows += int(overflowed)
        except Exception as error:
            self.error = error
        finally:
            self._stream = None
            self._metadata.update(
                status="failed" if self.error else ("complete" if self.frames else "empty"),
                frames=self.frames,
                duration_seconds=self.frames / self.rate,
                overflow_count=self.overflows,
                stopped_at=datetime.now(timezone.utc).isoformat(),
                error=str(self.error) if self.error else None,
            )
            try:
                write_metadata(self.path.with_suffix(".json"), self._metadata)
            except Exception as error:
                self.error = error
            self.ready.set()

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2)
            if self.running and self._stream is not None:
                self._stream.abort()
                self._thread.join(timeout=2)
            if self.running:
                raise RuntimeError(
                    "Microphone has not stopped yet. Keep the kernel open and retry Stop."
                )
        return self.path


def preview_recording(path, seconds=5):
    # Preview a bounded prefix rather than reading an hours-long recording into memory.
    info = sf.info(path)
    audio, rate = sf.read(
        path,
        frames=min(info.frames, round(seconds * info.samplerate)),
        dtype="float32",
        always_2d=True,
    )
    if not len(audio):
        return
    audio = audio.mean(axis=1)
    if rate != SAMPLE_RATE:
        factor = gcd(rate, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // factor, rate // factor)
    show_audio(
        audio.astype(np.float32), f"Recording preview: first {min(seconds, info.duration):.1f}s"
    )


previous_recorder = globals().get("_recorder") or globals().get("_noise_recorder")
if previous_recorder is not None and previous_recorder.running:
    raise RuntimeError("Stop recording before rerunning the control panel.")
previous_status_task = globals().get("_recording_status_task")
if previous_status_task is not None:
    previous_status_task.cancel()
previous_panel = globals().get("_pilot_panel")
if previous_panel is not None:
    close_panel(previous_panel)
_recorder = AudioRecorder()
_recording_status_task = None

input_options = [("System default microphone", None)] + [
    (f"{i}: {d['name']}", i)
    for i, d in enumerate(sd.query_devices())
    if d["max_input_channels"] > 0
]
device = widgets.Dropdown(
    options=input_options, description="Microphone:", layout={"width": "650px"}
)
window = widgets.Dropdown(options=[1.5, 2.0, 2.5, 3.0], value=2.0, description="Window (s):")
source_picker = widgets.Dropdown(description="Positive:", layout={"width": "650px"})
selection = widgets.FloatRangeSlider(
    value=(0, 1.0),
    min=0,
    max=1.0,
    step=0.01,
    description="Phrase (s):",
    readout_format=".2f",
    continuous_update=False,
    layout={"width": "650px"},
)
record_button = widgets.Button(description="Start positive", button_style="success")
negative_button = widgets.Button(description="Start negative")
hard_negative_button = widgets.Button(description="Start hard negative", layout={"width": "180px"})
hard_negative_stop_button = widgets.Button(
    description="Stop hard negative",
    button_style="danger",
    disabled=True,
    layout={"width": "180px"},
)
record_stop_button = widgets.Button(
    description="Stop positive", button_style="danger", disabled=True
)
negative_stop_button = widgets.Button(
    description="Stop negative", button_style="danger", disabled=True
)
noise_button = widgets.Button(description="Start noise", button_style="success")
noise_stop_button = widgets.Button(description="Stop noise", button_style="danger", disabled=True)
recording_status = widgets.HTML(value="Recorder idle. Click Start, then Stop when finished.")
preview_button = widgets.Button(description="Preview selection")
augment_button = widgets.Button(description="Generate 6 variants", button_style="info")
batch_button = widgets.Button(description="Augment all positives")
negative_aug_button = widgets.Button(description="Augment selected", layout={"width": "220px"})
negative_aug_scope = widgets.Dropdown(
    options=[("Selected processed files", "selected"), ("All processed files in Category", "all")],
    value="selected",
    description="Augment:",
    layout={"width": "550px"},
)
negative_processed = widgets.SelectMultiple(
    options=[],
    value=(),
    description="Files:",
    rows=7,
    layout={"width": "750px"},
)
negative_processed_refresh = widgets.Button(description="Refresh file list")
negative_aug_summary = widgets.HTML()
negative_window = widgets.Dropdown(
    options=window.options, value=window.value, description="Output (s):"
)
negative_summary = widgets.HTML()
negative_multiple = widgets.Checkbox(value=False, description="Use several time intervals")
negative_split_info = widgets.HTML(value="Split is locked for this source after the first save.")
negative_kind = widgets.Dropdown(
    options=[("Negative", "negative"), ("Hard negative", "hard_negative")], description="Category:"
)
negative_source = widgets.Dropdown(description="Source:", layout={"width": "650px"})
negative_info = widgets.HTML(value="Choose a completed negative recording.")
negative_ranges = widgets.Textarea(
    description="Ranges (s):",
    placeholder="One start,end pair per line, e.g. 10,30",
    layout={"width": "650px", "height": "90px"},
)
negative_mode = widgets.Dropdown(
    options=[("Random crops from recording", "random"), ("Slide through selection", "sliding")],
    description="Method:",
    layout={"width": "450px"},
)
negative_count = widgets.BoundedIntText(value=20, min=1, max=10000, description="Samples:")
negative_stride = widgets.FloatText(value=2.0, description="Stride (s):")
negative_split = widgets.Dropdown(
    options=["unassigned", "train", "validation", "test"], description="Split:"
)
negative_selection = widgets.FloatRangeSlider(
    value=(0, 0),
    min=0,
    max=1,
    step=0.01,
    description="Selection (s):",
    readout_format=".2f",
    continuous_update=False,
    disabled=True,
    layout={"width": "650px"},
    style={"description_width": "100px"},
)
negative_preview_button = widgets.Button(description="Preview selection", layout={"width": "180px"})
negative_use_range_button = widgets.Button(
    description="Replace interval list", layout={"width": "180px"}
)
negative_add_range_button = widgets.Button(
    description="Add selection to list", layout={"width": "180px"}
)
extract_button = widgets.Button(
    description="Save samples", button_style="info", layout={"width": "200px"}
)
legacy_button = widgets.Button(description="Import old recordings")
output = widgets.Output()
buttons = [
    record_button,
    record_stop_button,
    negative_button,
    negative_stop_button,
    hard_negative_button,
    hard_negative_stop_button,
    noise_button,
    noise_stop_button,
    preview_button,
    augment_button,
    batch_button,
    negative_aug_button,
    negative_processed_refresh,
    negative_preview_button,
    negative_use_range_button,
    negative_add_range_button,
    extract_button,
    legacy_button,
]


all_controls = [
    *buttons,
    device,
    window,
    source_picker,
    selection,
    negative_kind,
    negative_source,
    negative_ranges,
    negative_mode,
    negative_count,
    negative_stride,
    negative_split,
    negative_selection,
    negative_window,
    negative_multiple,
    negative_aug_scope,
    negative_processed,
]
negative_picker_state = {"updating": False}
negative_ui_state = {"updating": False}


stop_buttons = {
    "positive": record_stop_button,
    "negative": negative_stop_button,
    "hard_negative": hard_negative_stop_button,
    "noise": noise_stop_button,
}


def update_controls():
    recording = _recorder.running
    for control in all_controls:
        control.disabled = recording
    for control in [
        negative_selection,
        negative_preview_button,
        negative_use_range_button,
        negative_add_range_button,
    ]:
        control.disabled = recording or negative_source.value is None
    update_negative_preparation()
    update_processed_augmentation()
    for kind, button in stop_buttons.items():
        button.disabled = not (recording and _recorder.kind == kind)


def run_action(action):
    for control in all_controls:
        control.disabled = True
    output.outputs = ()
    try:
        action()
    except Exception as error:
        log(f"{type(error).__name__}: {error}")
    finally:
        update_controls()


async def monitor_recording():
    while _recorder.running:
        if _recorder.ready.is_set():
            elapsed = _recorder.frames / _recorder.rate
            recording_status.value = (
                f"Recording {_recorder.kind}: {elapsed:.1f} seconds. "
                f"Click Stop {_recorder.kind} to finish."
            )
        await asyncio.sleep(0.25)
    update_controls()
    if _recorder.error:
        recording_status.value = "Recording failed; the partial file is excluded from processing."
        log("Recording error:", _recorder.error)


def start_recording(kind):
    global _recording_status_task
    if _recording_status_task is not None:
        _recording_status_task.cancel()
    path = _recorder.start(device.value, kind)
    recording_status.value = (
        f"Opening microphone for {kind}; no countdown. Click Stop {kind} to finish."
    )
    log("Recording to:", path.relative_to(ROOT))
    log("Click Stop", kind, "when finished. There is no preset duration.")
    if kind == "positive":
        log("Say the complete Hey GPT phrase once. No automatic trimming is applied.")
    elif kind == "hard_negative":
        log(
            "Record a similar-sounding phrase or known false trigger, without the complete wake phrase."
        )
    else:
        log("Avoid saying the complete wake phrase in negative or noise recordings.")
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        pass
    else:
        _recording_status_task = loop.create_task(monitor_recording())


def stop_recording():
    global _recording_status_task
    if _recording_status_task is not None:
        _recording_status_task.cancel()
    path = _recorder.stop()
    duration = _recorder.frames / _recorder.rate
    kind = _recorder.kind
    recording_status.value = f"{kind.title()} saved: {duration:.2f} seconds."
    log("Saved original:", path.relative_to(ROOT))
    if _recorder.error:
        recording_status.value = "Recording failed; the partial file is excluded from processing."
        log("Recording error:", _recorder.error)
        return
    if _recorder.frames == 0:
        log("No audio was captured; this empty recording is excluded from processing.")
        return
    if _recorder.overflows:
        log(
            "Warning: input overflow occurred",
            _recorder.overflows,
            "time(s); review the recording.",
        )
    if kind == "positive":
        refresh_sources(path)
        load_positive(path)
    elif kind in {"negative", "hard_negative"}:
        negative_kind.value = kind
        refresh_negative_sources(path)
        log("Raw saved. Drag Selection, preview it, then click Save samples.")
        log("Drag Selection and click Preview selection to listen to the chosen interval.")
    else:
        preview_recording(path)


def refresh_sources(selected=None):
    state["updating_picker"] = True
    try:
        sources = available_audio_files(dataset_path("positive", "raw"))
        source_picker.options = [("Select a recording", None)] + [(p.name, str(p)) for p in sources]
        if selected is not None:
            source_picker.value = str(selected)
    finally:
        state["updating_picker"] = False


def load_positive(source):
    source = Path(source)
    audio = normalize_source(source)
    processed = process_recording("positive", source)[0]
    metadata = json.loads(processed.with_suffix(".json").read_text())
    state.update(audio=audio, source=source)
    selection.value = (0, 0)
    selection.max = len(audio) / SAMPLE_RATE
    selection.value = metadata["selection_seconds"]
    log("Raw source:", source.relative_to(ROOT))
    log("Processed phrase:", processed.relative_to(ROOT))
    show_audio(audio, "Original timing, normalized for preview")
    log(
        "The full recording is selected unless you saved a manual selection. Adjust Phrase only if needed."
    )


def selected_phrase():
    if state["audio"] is None:
        raise ValueError("Record or select a positive sample first.")
    start, end = selection.value
    phrase = state["audio"][round(start * SAMPLE_RATE) : round(end * SAMPLE_RATE)]
    if not len(phrase):
        raise ValueError("Select a nonempty phrase interval.")
    return phrase


def generate_variants():
    phrase = selected_phrase()
    if len(phrase) > round(window.value * SAMPLE_RATE):
        raise ValueError(
            "Phrase exceeds the window. Adjust the selection or increase the window; no audio was truncated."
        )
    processed = process_recording("positive", state["source"], bounds=selection.value)[0]
    show_audio(phrase, "Selected phrase")
    augment_recording("positive", processed, window_seconds=window.value, show_players=True)


def augment_all(kind):
    if kind in {"negative", "hard_negative"}:
        outputs = processed_negative_windows(kind)
        if not outputs:
            raise ValueError(
                "No saved samples found. Select a source and click Save samples first."
            )
        for processed in outputs:
            augment_recording(kind, processed)
        log(
            "Augmented",
            len(outputs),
            "existing",
            kind,
            "windows; no raw recording was reprocessed.",
        )
        return
    sources = available_audio_files(dataset_path(kind, "raw"))
    if not sources:
        raise ValueError(f"No {kind} recordings found.")
    count = 0
    for source in sources:
        outputs = process_recording(kind, source, window_seconds=window.value)
        for processed in outputs:
            if sf.info(processed).frames > round(window.value * SAMPLE_RATE):
                log(
                    "Skipped oversized phrase:",
                    source.name,
                    "- review its bounds or increase the window.",
                )
                continue
            augment_recording(kind, processed, window_seconds=window.value)
            count += 1
    log("Augmented", count, "processed positive file(s). Saved selections were reused.")


def refresh_processed_samples(selected=None):
    paths = processed_negative_windows(negative_kind.value)
    previous = negative_processed.value if selected is None else tuple(str(p) for p in selected)
    base = dataset_path(negative_kind.value, "processed")
    negative_processed.options = [(str(p.relative_to(base)), str(p)) for p in paths]
    allowed = {str(p) for p in paths}
    negative_processed.value = tuple(value for value in previous if value in allowed)
    if selected is not None:
        negative_aug_scope.value = "selected"
    update_processed_augmentation()


def update_processed_augmentation(change=None):
    recording = _recorder.running
    available = len(negative_processed.options)
    selected = len(negative_processed.value)
    all_files = negative_aug_scope.value == "all"
    count = available if all_files else selected
    negative_processed.disabled = recording or all_files
    negative_processed_refresh.disabled = recording
    negative_aug_button.disabled = recording or count == 0
    negative_aug_button.description = (
        f"Augment all {count} files" if all_files else f"Augment {count} selected"
    )
    if all_files:
        negative_aug_summary.value = f"All {available} processed files in {negative_kind.value}, across every raw source. Each click creates new augmentation runs."
    else:
        negative_aug_summary.value = f"{selected} of {available} processed files selected. Only highlighted files will be augmented. Save samples selects the newly saved files automatically."


def augment_chosen_negatives():
    kind = negative_kind.value
    available = {str(p): p for p in processed_negative_windows(kind)}
    if negative_aug_scope.value == "all":
        chosen = list(available.values())
    else:
        selected = negative_processed.value
        if any(value not in available for value in selected):
            raise ValueError(
                "A selected file is no longer available. Refresh the file list and select again."
            )
        chosen = [available[value] for value in selected]
    if not chosen:
        raise ValueError(
            "Select at least one processed file, or choose All processed files in Category."
        )
    for path in chosen:
        augment_recording(kind, path)
    log("Augmented", len(chosen), kind, "processed file(s).")


def refresh_negative_sources(selected=None):
    negative_picker_state["updating"] = True
    try:
        sources = available_audio_files(dataset_path(negative_kind.value, "raw"))
        negative_source.options = [("Select a recording", None)] + [
            (p.name, str(p)) for p in sources
        ]
        if selected is not None:
            negative_source.value = str(selected)
    finally:
        negative_picker_state["updating"] = False
    load_negative_source()


def load_negative_source():
    negative_ui_state["updating"] = True
    negative_ranges.value = ""
    negative_multiple.value = False
    negative_selection.value = (0, 0)
    negative_selection.max = 1
    if negative_source.value is None:
        negative_info.value = "Choose a completed negative recording."
        negative_split.value = "unassigned"
        negative_ui_state["updating"] = False
        update_controls()
        return
    source = Path(negative_source.value)
    info = sf.info(source)
    sidecar = source.with_suffix(".json")
    metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
    negative_split.value = metadata.get("split", "unassigned")
    negative_info.value = (
        f"Recording: {info.duration:.2f} seconds. Drag the handles to choose a section."
    )
    negative_selection.max = info.duration
    negative_selection.value = (0, info.duration)
    negative_mode.value = "placement" if negative_kind.value == "hard_negative" else "random"
    negative_count.max = 10000
    negative_count.value = 1 if negative_kind.value == "hard_negative" else 20
    negative_ui_state["updating"] = False
    log("Selected", source.name, f"({info.duration:.3f}s).")
    update_controls()


def change_negative_category(change):
    negative_mode.options = (
        [
            ("Random position in output window", "placement"),
            ("Centered in output window", "clips"),
        ]
        if change["new"] == "hard_negative"
        else [("Random crops from recording", "random"), ("Slide through selection", "sliding")]
    )
    negative_mode.value = "placement" if change["new"] == "hard_negative" else "random"
    refresh_negative_sources()
    refresh_processed_samples(selected=[])


def parse_negative_ranges():
    intervals = []
    for line in negative_ranges.value.splitlines():
        if not line.strip():
            continue
        try:
            start, end = map(float, line.split(","))
        except ValueError as error:
            raise ValueError(
                "Use one start,end pair in seconds per line, for example 10,30."
            ) from error
        intervals.append((start, end))
    return intervals


def negative_extraction_intervals():
    if negative_multiple.value:
        return parse_negative_ranges()
    path, info, start, end, first, last = selected_negative_range()
    return [(start, end)]


def extract_selected_negatives():
    if negative_source.value is None:
        raise ValueError("Select a negative source first.")
    saved = extract_negative_windows(
        negative_kind.value,
        negative_source.value,
        negative_extraction_intervals(),
        window_seconds=window.value,
        mode=negative_mode.value,
        count=negative_count.value,
        stride_seconds=negative_stride.value,
        split=negative_split.value,
    )

    refresh_processed_samples(selected=saved)


def update_negative_preparation(change=None):
    if negative_ui_state["updating"]:
        return
    negative_ui_state["updating"] = True
    try:
        recording = _recorder.running
        method = negative_mode.value
        negative_count.layout.display = "" if method in {"random", "placement"} else "none"
        negative_count.description = "Copies / clip:" if method == "placement" else "Samples:"
        negative_count.style.description_width = "100px" if method == "placement" else "initial"
        if method == "placement":
            negative_count.max = 10000
        negative_stride.layout.display = "" if method == "sliding" else "none"
        negative_ranges_box.layout.display = "" if negative_multiple.value else "none"
        for control in [negative_ranges, negative_use_range_button, negative_add_range_button]:
            control.disabled = (
                recording or negative_source.value is None or not negative_multiple.value
            )
        try:
            if negative_source.value is None:
                raise ValueError("Choose a recording above to prepare samples.")
            info = sf.info(negative_source.value)
            sidecar = Path(negative_source.value).with_suffix(".json")
            source_metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
            locked_split = source_metadata.get("window_split")
            if locked_split is not None:
                negative_split.value = locked_split
                negative_split.disabled = True
                negative_split_info.value = (
                    f"This recording is locked to {locked_split}; all its samples use that split."
                )
            else:
                negative_split.disabled = recording
                negative_split_info.value = "Choose a split before the first save; it will then be locked for this recording."
            intervals = negative_extraction_intervals()
            if not intervals:
                raise ValueError(
                    "Add at least one interval in Advanced, or turn off several intervals to use the slider."
                )
            frames = []
            for start, end in sorted(intervals):
                if not (
                    np.isfinite(start) and np.isfinite(end) and 0 <= start < end <= info.duration
                ):
                    raise ValueError("Choose a nonempty section inside the recording.")
                first = int(np.ceil(start * info.samplerate))
                last = info.frames if end == info.duration else int(np.floor(end * info.samplerate))
                if last <= first or round((last - first) * SAMPLE_RATE / info.samplerate) < 1:
                    raise ValueError("Choose a section containing at least one audio sample.")
                if frames and first < frames[-1][1]:
                    raise ValueError("Time intervals overlap. Remove the overlap in Advanced.")
                frames.append((first, last))
            target = round(round(window.value * SAMPLE_RATE) * info.samplerate / SAMPLE_RATE)
            sizes = [b - a for a, b in frames]
            duration = sum(sizes) / info.samplerate
            prefix = f"Selected: {duration:.2f}s. Each saved sample: {window.value:.2f}s. "
            if method in {"clips", "placement"}:
                if any(size > target for size in sizes):
                    raise ValueError(prefix + "Select a shorter clip or increase Output (s).")
                padding = sum(max(0, target - size) for size in sizes) / info.samplerate
                if method == "placement":
                    total = len(sizes) * negative_count.value
                    message = (
                        prefix
                        + f"Keep each complete clip. Save {negative_count.value} random placement(s) per clip ({total} total). Fill the rest with silence."
                    )
                    if all(size == target for size in sizes):
                        message += " The clip fills the output window, so its position cannot move."
                else:
                    total = len(sizes)
                    message = (
                        prefix
                        + f"Save {total} centered sample(s); add {padding:.2f}s of silence in total."
                    )
                extract_button.description = (
                    "Save 1 sample" if total == 1 else f"Save {total} samples"
                )
            else:
                if any(size < target for size in sizes):
                    raise ValueError(prefix + "Select a longer section or reduce Output (s).")
                if method == "random":
                    capacity = sum(size - target + 1 for size in sizes)
                    negative_count.max = min(10000, capacity)
                    message = (
                        prefix
                        + f"Save {negative_count.value} random sample(s) inside the selected section(s). Overlap is allowed."
                    )
                else:
                    stride = negative_stride.value
                    if (
                        not np.isfinite(stride)
                        or stride <= 0
                        or round(stride * info.samplerate) < 1
                    ):
                        raise ValueError("Set a positive Stride in seconds.")
                    total = sum(
                        (size - target) // round(stride * info.samplerate) + 1 for size in sizes
                    )
                    message = prefix + f"Save {total} sample(s), moving {stride:.2f}s each time."
                extract_button.description = "Save samples"
            negative_summary.value = message
            extract_button.disabled = recording
        except ValueError as error:
            negative_summary.value = str(error)
            extract_button.disabled = True
    finally:
        negative_ui_state["updating"] = False


def toggle_multiple_intervals(change):
    if negative_ui_state["updating"]:
        return
    if change["new"] and not negative_ranges.value.strip() and negative_source.value is not None:
        start, end = negative_selection.value
        negative_ranges.value = f"{start!r},{end!r}"
    update_negative_preparation()


def sync_output_window(change):
    other = negative_window if change["owner"] is window else window
    if other.value != change["new"]:
        other.value = change["new"]
    update_negative_preparation()


def selected_negative_range():
    if negative_source.value is None:
        raise ValueError("Select a negative source first.")
    path = Path(negative_source.value)
    info = sf.info(path)
    start, end = negative_selection.value
    if not (np.isfinite(start) and np.isfinite(end) and 0 <= start < end <= info.duration):
        raise ValueError("Select a nonempty interval inside the recording.")
    first = int(np.ceil(start * info.samplerate))
    last = info.frames if end == info.duration else int(np.floor(end * info.samplerate))
    if last <= first or round((last - first) * SAMPLE_RATE / info.samplerate) < 1:
        raise ValueError("Select an interval containing at least one audio sample.")
    return path, info, start, end, first, last


def preview_negative_source():
    path, info, start, end, first, last = selected_negative_range()
    audio = read_normalized_segment(path, first, last - first)
    show_audio(audio, f"Selected source interval: {start:.3f}-{end:.3f}s")


def use_negative_range(append=False):
    path, info, start, end, first, last = selected_negative_range()
    intervals = parse_negative_ranges() if append else []
    intervals.append((start, end))
    intervals.sort()
    if any(
        not (np.isfinite(a) and np.isfinite(b) and 0 <= a < b <= info.duration)
        for a, b in intervals
    ):
        raise ValueError("Every range must be a nonempty interval inside this source.")
    if any(current[0] < previous[1] for previous, current in zip(intervals, intervals[1:])):
        raise ValueError(
            "Ranges overlap. Use range to replace the list, or select a separate interval."
        )
    negative_ranges.value = "\n".join(f"{a!r},{b!r}" for a, b in intervals)
    negative_multiple.value = True
    log("Interval list updated:", len(intervals), "interval(s).")
    update_negative_preparation()


def import_old():
    imported = import_legacy_recordings()
    refresh_sources(state["source"])
    log("Imported", len(imported), "recordings. Old files were preserved.")


def on_source_change(change):
    if change["new"] is not None and not state["updating_picker"]:
        run_action(lambda: load_positive(change["new"]))


def action_row(controls, explanation):
    return widgets.HBox(
        [
            *controls,
            widgets.HTML(
                value=f"<span style='color:var(--vscode-descriptionForeground, #666)'>{explanation}</span>",
                layout={"flex": "1 1 320px"},
            ),
        ],
        layout={"align_items": "center", "flex_flow": "row wrap"},
    )


negative_ranges_box = widgets.VBox(
    [
        widgets.HTML(
            value="With several intervals enabled, this list replaces the slider for saving samples."
        ),
        negative_ranges,
        action_row(
            [negative_use_range_button], "Replace the interval list with the slider selection."
        ),
        action_row(
            [negative_add_range_button], "Append the slider selection to the interval list."
        ),
    ]
)
negative_advanced = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                negative_split,
                negative_split_info,
                negative_multiple,
                negative_ranges_box,
            ]
        )
    ],
    selected_index=None,
)
negative_advanced.set_title(0, "Advanced: dataset split and several intervals")
legacy_tools = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                action_row(
                    [legacy_button],
                    "Copy recordings from the old pilot folders into the current layout.",
                )
            ]
        )
    ],
    selected_index=None,
)
legacy_tools.set_title(0, "Legacy data tools")


refresh_sources()
source_picker.observe(on_source_change, names="value")
refresh_negative_sources()
refresh_processed_samples()
negative_kind.observe(change_negative_category, names="value")
negative_source.observe(
    lambda change: (
        run_action(load_negative_source) if not negative_picker_state["updating"] else None
    ),
    names="value",
)
for control in [
    negative_selection,
    negative_mode,
    negative_count,
    negative_stride,
    negative_ranges,
    negative_split,
]:
    control.observe(update_negative_preparation, names="value")
negative_multiple.observe(toggle_multiple_intervals, names="value")
window.observe(sync_output_window, names="value")
negative_window.observe(sync_output_window, names="value")
extract_button.on_click(lambda _: run_action(extract_selected_negatives))
negative_preview_button.on_click(lambda _: run_action(preview_negative_source))
negative_use_range_button.on_click(lambda _: run_action(use_negative_range))
negative_add_range_button.on_click(lambda _: run_action(lambda: use_negative_range(append=True)))
hard_negative_button.on_click(lambda _: run_action(lambda: start_recording("hard_negative")))
hard_negative_stop_button.on_click(lambda _: run_action(stop_recording))
record_button.on_click(lambda _: run_action(lambda: start_recording("positive")))
record_stop_button.on_click(lambda _: run_action(stop_recording))
negative_button.on_click(lambda _: run_action(lambda: start_recording("negative")))
negative_stop_button.on_click(lambda _: run_action(stop_recording))
noise_button.on_click(lambda _: run_action(lambda: start_recording("noise")))
noise_stop_button.on_click(lambda _: run_action(stop_recording))
preview_button.on_click(
    lambda _: run_action(lambda: show_audio(selected_phrase(), "Selected phrase"))
)
augment_button.on_click(lambda _: run_action(generate_variants))
batch_button.on_click(lambda _: run_action(lambda: augment_all("positive")))
negative_aug_button.on_click(lambda _: run_action(augment_chosen_negatives))
negative_processed_refresh.on_click(lambda _: run_action(refresh_processed_samples))
negative_processed.observe(update_processed_augmentation, names="value")
negative_aug_scope.observe(update_processed_augmentation, names="value")
legacy_button.on_click(lambda _: run_action(import_old))
_pilot_panel = widgets.VBox(
    [
        device,
        window,
        source_picker,
        selection,
        action_row(
            [record_button, record_stop_button],
            "Record a positive example. Stop saves raw and prepares the selected positive source.",
        ),
        action_row(
            [negative_button, negative_stop_button],
            "Record ordinary negatives. Stop saves raw; use Save samples below to process it.",
        ),
        action_row(
            [hard_negative_button, hard_negative_stop_button],
            "Record a hard-negative example. Stop saves raw; use Save samples below to process it.",
        ),
        action_row(
            [noise_button, noise_stop_button],
            "Record background noise into noise/raw/ for mixing into augmented samples.",
        ),
        recording_status,
        action_row([preview_button], "Listen to the selected positive phrase."),
        action_row(
            [augment_button],
            "Save the selected phrase and create 6 augmented variants for this positive source.",
        ),
        action_row(
            [batch_button],
            "Create variants for ALL positive raw recordings, using their saved phrase selections.",
        ),
        widgets.HTML(value="<b>Prepare negative samples</b>"),
        negative_kind,
        negative_source,
        negative_info,
        negative_selection,
        action_row(
            [negative_preview_button],
            "Listen to the current slider selection. Save samples creates the processed audio.",
        ),
        negative_window,
        negative_mode,
        negative_count,
        negative_stride,
        negative_summary,
        negative_advanced,
        action_row(
            [extract_button],
            "Selected Source + slider interval (or the enabled Advanced interval list) → processed/. Each click saves a new set.",
        ),
        widgets.HTML(value="<b>Augment processed audio</b>"),
        negative_aug_scope,
        negative_processed,
        action_row(
            [negative_processed_refresh],
            "File labels include the source and extraction folder. Ctrl/Cmd-click for several files; Shift-click for a range.",
        ),
        widgets.HBox(
            [negative_aug_button, negative_aug_summary],
            layout={"align_items": "center", "flex_flow": "row wrap"},
        ),
        legacy_tools,
        output,
    ]
)
display(_pilot_panel)
update_controls()
log("Ready. Recording starts only when you click a recording button.")
log("Noise is shared across sessions from data/noise/raw/.")